In [1]:
# ============================================================
# PHASE 4A — MULTILEVEL / HIERARCHICAL ANALYSIS AUDIT
# ============================================================
#
# PURPOSE
# -------
# Before fitting system-specific weighted regressions, audit:
#
#   1. usable sample sizes
#   2. focal-predictor missingness
#   3. complete-case attrition
#   4. school-cluster counts
#   5. effective weighted sample sizes
#   6. subtype event counts
#   7. gender coding / support
#   8. availability of Stage-2 system moderators
#
#
# FOCAL PREDICTORS
# ----------------
#
#   BELONG      = school belonging
#   DISCLIMA    = disciplinary climate
#   ST004D01T   = gender
#
#
# THREE SAMPLE SPECIFICATIONS ARE AUDITED
# ----------------------------------------
#
# Focal3:
#   BELONG + DISCLIMA + gender
#
# Minimal5:
#   focal3 + AGE + ESCS
#
# Core14:
#   complete cases on all original 14 predictors
#
#
# IMPORTANT
# ---------
# This cell does NOT decide which adjustment set is final.
# It tells us what each choice would cost in sample size.
#
# NO REGRESSION FITTING.
# NO IMPUTATION.
# NO META-REGRESSION.
# ============================================================


from pathlib import Path
import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RANKING_PATH = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


MACRO_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "macro_validators_with_derived_groups.csv"
)


# ============================================================
# 2. Frozen analysis variables
# ============================================================

OUTCOMES = [

    "BEINGBULLIED",

    "BULLY_VERBAL",

    "BULLY_THREAT",

    "BULLY_PHYSICAL",

    "BULLY_RELATIONAL",
]


FOCAL3 = [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",
]


MINIMAL5 = [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",

    "AGE",

    "ESCS",
]


CORE14 = [

    "ADAPTIVITY",

    "AGE",

    "ATTLNACT",

    "BELONG",

    "DISCLIMA",

    "EDUSHORT",

    "ESCS",

    "JOYREAD",

    "MASTGOAL",

    "METASPAM",

    "METASUM",

    "ST004D01T",

    "STIMREAD",

    "UNDREM",
]


MODEL_SPECS = {

    "Focal3":
        FOCAL3,

    "Minimal5":
        MINIMAL5,

    "Core14":
        CORE14,
}


SCHOOL_COL = "CNTSCHID"

STUDENT_COL = "CNTSTUID"

WEIGHT_COL = "W_FSTUWT"


# ============================================================
# 3. Recover exact frozen international system universe
#
# 74 systems have BEINGBULLIED.
# 75 systems have subtype outcomes.
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


SYSTEMS = sorted(
    rankings[
        "Country"
    ]
    .dropna()
    .astype(
        str
    )
    .unique()
)


print(
    "Frozen international system universe:",
    len(
        SYSTEMS
    )
)


print(
    SYSTEMS
)


# ============================================================
# 4. Numeric helpers
# ============================================================

def numeric_series(
    series
):

    output = pd.to_numeric(
        series,
        errors="coerce"
    )


    output = output.astype(
        float
    )


    output[
        ~np.isfinite(
            output
        )
    ] = np.nan


    return output


def kish_effective_n(
    weights
):

    weights = np.asarray(
        weights,
        dtype=float
    )


    weights = weights[
        np.isfinite(
            weights
        )
        &
        (
            weights
            >
            0
        )
    ]


    if len(
        weights
    ) == 0:

        return np.nan


    denominator = np.sum(
        weights ** 2
    )


    if denominator <= 0:

        return np.nan


    return float(
        (
            np.sum(
                weights
            )
            **
            2
        )
        /
        denominator
    )


def weighted_mean(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    mask = (

        np.isfinite(
            values
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )
    )


    if mask.sum() == 0:

        return np.nan


    return float(
        np.average(
            values[
                mask
            ],
            weights=weights[
                mask
            ]
        )
    )


def weighted_sd(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    mask = (

        np.isfinite(
            values
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )
    )


    if mask.sum() < 2:

        return np.nan


    values = values[
        mask
    ]


    weights = weights[
        mask
    ]


    mean = np.average(
        values,
        weights=weights
    )


    variance = np.average(
        (
            values
            -
            mean
        )
        **
        2,
        weights=weights
    )


    return float(
        np.sqrt(
            variance
        )
    )


# ============================================================
# 5. Audit containers
# ============================================================

variable_rows = []

sample_rows = []

gender_rows = []

school_rows = []


# ============================================================
# 6. System × outcome audit
# ============================================================

for system_index, country in enumerate(
    SYSTEMS,
    start=1
):

    path = (
        PROCESSED_DIR
        /
        f"{country}_analysis.pkl"
    )


    if not path.exists():

        raise FileNotFoundError(
            f"Missing processed file: {path}"
        )


    df = pd.read_pickle(
        path
    )


    print(
        f"[{system_index:02d}/{len(SYSTEMS)}] "
        f"{country}"
    )


    # --------------------------------------------------------
    # Schema QA
    # --------------------------------------------------------

    required_columns = (

        [
            SCHOOL_COL,
            STUDENT_COL,
            WEIGHT_COL,
        ]

        +

        OUTCOMES

        +

        CORE14
    )


    missing_columns = [
        column
        for column
        in required_columns
        if column
        not in df.columns
    ]


    if len(
        missing_columns
    ) > 0:

        raise RuntimeError(
            f"{country} missing columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------
    # Standardized numeric copies
    # --------------------------------------------------------

    weight = numeric_series(
        df[
            WEIGHT_COL
        ]
    )


    numeric_predictors = {

        variable:
            numeric_series(
                df[
                    variable
                ]
            )

        for variable
        in CORE14
    }


    base_valid = (

        weight.notna()

        &

        (
            weight
            >
            0
        )

        &

        df[
            SCHOOL_COL
        ]
        .notna()
    )


    # ========================================================
    # Outcome loop
    # ========================================================

    for outcome in OUTCOMES:

        y = numeric_series(
            df[
                outcome
            ]
        )


        outcome_valid = (

            base_valid

            &

            y.notna()
        )


        n_outcome_valid = int(
            outcome_valid.sum()
        )


        # KOR overall bullying is expected to be zero.
        if n_outcome_valid == 0:

            for spec_name in MODEL_SPECS:

                sample_rows.append(
                    {
                        "Country":
                            country,

                        "Outcome":
                            outcome,

                        "Specification":
                            spec_name,

                        "N_OutcomeValid":
                            0,

                        "N_CompleteCase":
                            0,

                        "RetentionPct":
                            np.nan,

                        "N_Schools":
                            0,

                        "KishEffectiveN":
                            np.nan,

                        "OutcomeUnique":
                            0,

                        "N_Positive":
                            np.nan,

                        "N_Negative":
                            np.nan,

                        "WeightedPrevalence":
                            np.nan,

                        "Flag_LowSchools":
                            True,

                        "Flag_LowEvents":
                            np.nan,

                        "Flag_LowRetention":
                            np.nan,
                    }
                )


            continue


        # ====================================================
        # Variable-level focal coverage
        # ====================================================

        for variable in FOCAL3:

            values = (
                numeric_predictors[
                    variable
                ]
            )


            valid_variable = (

                outcome_valid

                &

                values.notna()
            )


            n_valid = int(
                valid_variable.sum()
            )


            missing_rate = (

                1

                -

                (
                    n_valid
                    /
                    n_outcome_valid
                )
            )


            observed = values[
                valid_variable
            ]


            variable_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "Variable":
                        variable,

                    "N_OutcomeValid":
                        n_outcome_valid,

                    "N_Observed":
                        n_valid,

                    "MissingRate":
                        missing_rate,

                    "N_Unique":
                        int(
                            observed.nunique()
                        ),

                    "Min":
                        (
                            float(
                                observed.min()
                            )
                            if len(
                                observed
                            )
                            >
                            0
                            else np.nan
                        ),

                    "Max":
                        (
                            float(
                                observed.max()
                            )
                            if len(
                                observed
                            )
                            >
                            0
                            else np.nan
                        ),

                    "WeightedMean":
                        weighted_mean(
                            values[
                                valid_variable
                            ],
                            weight[
                                valid_variable
                            ]
                        ),

                    "WeightedSD":
                        weighted_sd(
                            values[
                                valid_variable
                            ],
                            weight[
                                valid_variable
                            ]
                        ),
                }
            )


        # ====================================================
        # Gender coding audit
        # ====================================================

        gender = (
            numeric_predictors[
                "ST004D01T"
            ]
        )


        gender_mask = (

            outcome_valid

            &

            gender.notna()
        )


        gender_counts = (

            gender[
                gender_mask
            ]
            .value_counts()
            .sort_index()
        )


        gender_rows.append(
            {
                "Country":
                    country,

                "Outcome":
                    outcome,

                "N_GenderObserved":
                    int(
                        gender_mask.sum()
                    ),

                "GenderValues":
                    "|".join(
                        [
                            str(
                                value
                            )
                            for value
                            in gender_counts.index
                        ]
                    ),

                "GenderCounts":
                    "|".join(
                        [
                            f"{value}:{count}"
                            for value, count
                            in gender_counts.items()
                        ]
                    ),

                "N_GenderLevels":
                    int(
                        len(
                            gender_counts
                        )
                    ),
            }
        )


        # ====================================================
        # Candidate inferential samples
        # ====================================================

        for spec_name, predictors in (
            MODEL_SPECS.items()
        ):

            complete_mask = (
                outcome_valid.copy()
            )


            for predictor in predictors:

                complete_mask &= (
                    numeric_predictors[
                        predictor
                    ]
                    .notna()
                )


            n_complete = int(
                complete_mask.sum()
            )


            retention = (

                n_complete
                /
                n_outcome_valid

                if n_outcome_valid
                >
                0

                else np.nan
            )


            schools = (
                df.loc[
                    complete_mask,
                    SCHOOL_COL
                ]
            )


            n_schools = int(
                schools.nunique()
            )


            sample_weights = (
                weight[
                    complete_mask
                ]
                .to_numpy(
                    dtype=float
                )
            )


            effective_n = (
                kish_effective_n(
                    sample_weights
                )
            )


            sample_y = (
                y[
                    complete_mask
                ]
            )


            outcome_unique = int(
                sample_y.nunique()
            )


            # ------------------------------------------------
            # Classification event support
            # ------------------------------------------------

            if outcome == "BEINGBULLIED":

                n_positive = np.nan

                n_negative = np.nan

                weighted_prevalence = np.nan

                low_events = np.nan


            else:

                n_positive = int(
                    (
                        sample_y
                        ==
                        1
                    )
                    .sum()
                )


                n_negative = int(
                    (
                        sample_y
                        ==
                        0
                    )
                    .sum()
                )


                if n_complete > 0:

                    weighted_prevalence = (
                        weighted_mean(
                            sample_y,
                            sample_weights
                        )
                    )


                else:

                    weighted_prevalence = np.nan


                # QA flag only.
                #
                # Not yet an exclusion criterion.
                low_events = (

                    n_positive
                    <
                    50

                    or

                    n_negative
                    <
                    50
                )


            sample_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "Specification":
                        spec_name,

                    "N_OutcomeValid":
                        n_outcome_valid,

                    "N_CompleteCase":
                        n_complete,

                    "RetentionPct":
                        (
                            retention
                            *
                            100
                        ),

                    "N_Schools":
                        n_schools,

                    "KishEffectiveN":
                        effective_n,

                    "OutcomeUnique":
                        outcome_unique,

                    "N_Positive":
                        n_positive,

                    "N_Negative":
                        n_negative,

                    "WeightedPrevalence":
                        weighted_prevalence,

                    # QA flags only
                    "Flag_LowSchools":
                        (
                            n_schools
                            <
                            30
                        ),

                    "Flag_LowEvents":
                        low_events,

                    "Flag_LowRetention":
                        (
                            retention
                            <
                            0.80
                        ),
                }
            )


            # ------------------------------------------------
            # School cluster-size QA
            # ------------------------------------------------

            if n_complete > 0:

                school_sizes = (

                    df.loc[
                        complete_mask,
                        SCHOOL_COL
                    ]
                    .value_counts()
                )


                school_rows.append(
                    {
                        "Country":
                            country,

                        "Outcome":
                            outcome,

                        "Specification":
                            spec_name,

                        "N_Schools":
                            n_schools,

                        "MedianStudentsPerSchool":
                            float(
                                school_sizes.median()
                            ),

                        "MinStudentsPerSchool":
                            int(
                                school_sizes.min()
                            ),

                        "MaxStudentsPerSchool":
                            int(
                                school_sizes.max()
                            ),

                        "SchoolsWith1Student":
                            int(
                                (
                                    school_sizes
                                    ==
                                    1
                                )
                                .sum()
                            ),
                    }
                )


    del df

    gc.collect()


# ============================================================
# 7. Convert to DataFrames
# ============================================================

variable_audit = pd.DataFrame(
    variable_rows
)


sample_audit = pd.DataFrame(
    sample_rows
)


gender_audit = pd.DataFrame(
    gender_rows
)


school_audit = pd.DataFrame(
    school_rows
)


# ============================================================
# 8. Save detailed audits
# ============================================================

variable_audit.to_csv(
    OUT_DIR
    / "stage1_focal_predictor_coverage.csv",
    index=False
)


sample_audit.to_csv(
    OUT_DIR
    / "stage1_complete_case_sample_audit.csv",
    index=False
)


gender_audit.to_csv(
    OUT_DIR
    / "stage1_gender_coding_audit.csv",
    index=False
)


school_audit.to_csv(
    OUT_DIR
    / "stage1_school_cluster_audit.csv",
    index=False
)


# ============================================================
# 9. Focal-variable international coverage summary
# ============================================================

variable_summary = (

    variable_audit
    .groupby(
        [
            "Outcome",
            "Variable"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "nunique"
        ),

        MedianMissingRate=(
            "MissingRate",
            "median"
        ),

        MaxMissingRate=(
            "MissingRate",
            "max"
        ),

        MinUnique=(
            "N_Unique",
            "min"
        ),

        MedianWeightedSD=(
            "WeightedSD",
            "median"
        ),

        MinWeightedSD=(
            "WeightedSD",
            "min"
        ),
    )
)


# ============================================================
# 10. Complete-case retention summary
# ============================================================

sample_summary = (

    sample_audit[
        sample_audit[
            "N_OutcomeValid"
        ]
        >
        0
    ]
    .groupby(
        [
            "Outcome",
            "Specification"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "nunique"
        ),

        MedianOutcomeN=(
            "N_OutcomeValid",
            "median"
        ),

        MedianCompleteN=(
            "N_CompleteCase",
            "median"
        ),

        MinCompleteN=(
            "N_CompleteCase",
            "min"
        ),

        MedianRetentionPct=(
            "RetentionPct",
            "median"
        ),

        MinRetentionPct=(
            "RetentionPct",
            "min"
        ),

        MedianSchools=(
            "N_Schools",
            "median"
        ),

        MinSchools=(
            "N_Schools",
            "min"
        ),

        MedianKishN=(
            "KishEffectiveN",
            "median"
        ),

        MinKishN=(
            "KishEffectiveN",
            "min"
        ),

        N_LowSchoolFlags=(
            "Flag_LowSchools",
            "sum"
        ),

        N_LowRetentionFlags=(
            "Flag_LowRetention",
            "sum"
        ),
    )
)


# ============================================================
# 11. Classification event summary
# ============================================================

classification_summary = (

    sample_audit[
        (
            sample_audit[
                "Outcome"
            ]
            !=
            "BEINGBULLIED"
        )
        &
        (
            sample_audit[
                "N_OutcomeValid"
            ]
            >
            0
        )
    ]
    .groupby(
        [
            "Outcome",
            "Specification"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "nunique"
        ),

        MedianPositive=(
            "N_Positive",
            "median"
        ),

        MinPositive=(
            "N_Positive",
            "min"
        ),

        MedianNegative=(
            "N_Negative",
            "median"
        ),

        MinNegative=(
            "N_Negative",
            "min"
        ),

        MinWeightedPrevalence=(
            "WeightedPrevalence",
            "min"
        ),

        MaxWeightedPrevalence=(
            "WeightedPrevalence",
            "max"
        ),

        N_LowEventFlags=(
            "Flag_LowEvents",
            "sum"
        ),
    )
)


# ============================================================
# 12. Worst complete-case attrition profiles
# ============================================================

lowest_retention = (

    sample_audit[
        sample_audit[
            "N_OutcomeValid"
        ]
        >
        0
    ]
    .sort_values(
        "RetentionPct"
    )
    [
        [
            "Country",
            "Outcome",
            "Specification",
            "N_OutcomeValid",
            "N_CompleteCase",
            "RetentionPct",
            "N_Schools",
            "KishEffectiveN",
            "N_Positive",
            "N_Negative",
        ]
    ]
    .head(
        30
    )
)


# ============================================================
# 13. Potential Stage-1 feasibility issues
# ============================================================

feasibility_flags = (

    sample_audit[
        (
            sample_audit[
                "N_OutcomeValid"
            ]
            >
            0
        )
        &
        (
            (
                sample_audit[
                    "Flag_LowSchools"
                ]
            )
            |
            (
                sample_audit[
                    "Flag_LowRetention"
                ]
            )
            |
            (
                sample_audit[
                    "Flag_LowEvents"
                ]
                .fillna(
                    False
                )
            )
        )
    ]
    .copy()
)


feasibility_flags.to_csv(
    OUT_DIR
    / "stage1_feasibility_flags.csv",
    index=False
)


# ============================================================
# 14. Gender coding consistency
# ============================================================

gender_pattern_summary = (

    gender_audit
    .groupby(
        "GenderValues",
        as_index=False
    )
    .agg(

        N_CountryOutcomeProfiles=(
            "Country",
            "count"
        ),

        N_Systems=(
            "Country",
            "nunique"
        ),
    )
    .sort_values(
        "N_CountryOutcomeProfiles",
        ascending=False
    )
)


# ============================================================
# 15. Stage-2 macro moderator audit
#
# Planned candidates:
#
#   - log GDP per capita PPP
#   - broad region
#   - Gini inequality
#
# No moderation model is fitted yet.
# ============================================================

macro_summary = pd.DataFrame()


if MACRO_PATH.exists():

    macro = pd.read_csv(
        MACRO_PATH
    )


    print("\nMacro columns available:")

    print(
        macro.columns.tolist()
    )


    # --------------------------------------------------------
    # Detect Gini column robustly
    # --------------------------------------------------------

    gini_candidates = [

        column
        for column
        in macro.columns
        if "GINI"
        in column.upper()
        and
        "GROUP"
        not in column.upper()
        and
        "TERT"
        not in column.upper()
    ]


    print(
        "\nPossible Gini columns:",
        gini_candidates
    )


    moderator_columns = []


    for candidate in [

        "LogGDPpcPPP_2018",

        "WB_Region",
    ]:

        if candidate in macro.columns:

            moderator_columns.append(
                candidate
            )


    if len(
        gini_candidates
    ) > 0:

        # Prefer columns that look like an actual numeric value.
        gini_column = gini_candidates[
            0
        ]


        moderator_columns.append(
            gini_column
        )


    macro_primary = (

        macro[
            macro[
                "Country"
            ]
            .isin(
                SYSTEMS
            )
        ]
        .copy()
    )


    macro_rows = []


    for variable in moderator_columns:

        values = (
            macro_primary[
                variable
            ]
        )


        macro_rows.append(
            {
                "Moderator":
                    variable,

                "N_Available":
                    int(
                        values.notna().sum()
                    ),

                "PctAvailable":
                    float(
                        values.notna().mean()
                        *
                        100
                    ),

                "NUnique":
                    int(
                        values.nunique(
                            dropna=True
                        )
                    ),
            }
        )


    macro_summary = pd.DataFrame(
        macro_rows
    )


    macro_summary.to_csv(
        OUT_DIR
        / "stage2_macro_moderator_coverage.csv",
        index=False
    )


else:

    print(
        "\nWARNING: macro validator file not found:"
    )

    print(
        MACRO_PATH
    )


# ============================================================
# 16. Save summaries
# ============================================================

variable_summary.to_csv(
    OUT_DIR
    / "stage1_focal_predictor_coverage_summary.csv",
    index=False
)


sample_summary.to_csv(
    OUT_DIR
    / "stage1_complete_case_summary.csv",
    index=False
)


classification_summary.to_csv(
    OUT_DIR
    / "stage1_classification_event_summary.csv",
    index=False
)


gender_pattern_summary.to_csv(
    OUT_DIR
    / "stage1_gender_pattern_summary.csv",
    index=False
)


# ============================================================
# 17. Print focal predictor coverage
# ============================================================

print("\n")
print("=" * 125)
print("FOCAL PREDICTOR COVERAGE")
print("=" * 125)


print(
    variable_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 18. Print complete-case sample comparison
# ============================================================

print("\n")
print("=" * 125)
print("COMPLETE-CASE RETENTION BY CANDIDATE STAGE-1 SPECIFICATION")
print("=" * 125)


print(
    sample_summary
    .round(
        2
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 19. Event support
# ============================================================

print("\n")
print("=" * 125)
print("CLASSIFICATION EVENT SUPPORT")
print("=" * 125)


print(
    classification_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 20. Worst attrition
# ============================================================

print("\n")
print("=" * 125)
print("LOWEST COMPLETE-CASE RETENTION PROFILES")
print("=" * 125)


print(
    lowest_retention
    .round(
        2
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Feasibility flags
# ============================================================

print("\n")
print("=" * 125)
print("STAGE-1 FEASIBILITY FLAGS")
print("=" * 125)


print(
    "Flagged profiles:",
    len(
        feasibility_flags
    )
)


if len(
    feasibility_flags
) == 0:

    print(
        "None."
    )


else:

    print(
        feasibility_flags[
            [
                "Country",
                "Outcome",
                "Specification",
                "N_CompleteCase",
                "RetentionPct",
                "N_Schools",
                "N_Positive",
                "N_Negative",
                "Flag_LowSchools",
                "Flag_LowEvents",
                "Flag_LowRetention",
            ]
        ]
        .round(
            2
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 22. Gender coding
# ============================================================

print("\n")
print("=" * 125)
print("GENDER CODING AUDIT")
print("=" * 125)


print(
    gender_pattern_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 23. Stage-2 moderator coverage
# ============================================================

print("\n")
print("=" * 125)
print("STAGE-2 MODERATOR COVERAGE")
print("=" * 125)


if len(
    macro_summary
) == 0:

    print(
        "No moderator coverage table created."
    )


else:

    print(
        macro_summary
        .round(
            2
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 24. Final status
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 4A MULTILEVEL AUDIT COMPLETE")
print("=" * 125)


print(
    "\nNo regression models have been fitted."
)


print(
    "\nNext decision:"
)


print(
    "Choose the Stage-1 adjustment set based on "
    "retention, event support, and scientific interpretation."
)


print(
    "\nThen we will fit survey-weighted system-specific "
    "linear/logistic models with school-clustered uncertainty."
)

Frozen international system universe: 75
['ALB', 'ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGR', 'BIH', 'BLR', 'BRA', 'BRN', 'CAN', 'CHE', 'CHL', 'COL', 'CRI', 'CZE', 'DEU', 'DNK', 'DOM', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GEO', 'GRC', 'HKG', 'HRV', 'HUN', 'IDN', 'IRL', 'ISL', 'ITA', 'JOR', 'JPN', 'KAZ', 'KOR', 'KSV', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MEX', 'MLT', 'MNE', 'MYS', 'NLD', 'NOR', 'NZL', 'PAN', 'PER', 'PHL', 'POL', 'PRT', 'QAT', 'QAZ', 'QCI', 'ROU', 'RUS', 'SAU', 'SGP', 'SRB', 'SVK', 'SVN', 'SWE', 'TAP', 'THA', 'TUR', 'UKR', 'URY', 'USA', 'VNM']
[01/75] ALB
[02/75] ARE
[03/75] ARG
[04/75] AUS
[05/75] AUT
[06/75] BEL
[07/75] BGR
[08/75] BIH
[09/75] BLR
[10/75] BRA
[11/75] BRN
[12/75] CAN
[13/75] CHE
[14/75] CHL
[15/75] COL
[16/75] CRI
[17/75] CZE
[18/75] DEU
[19/75] DNK
[20/75] DOM
[21/75] ESP
[22/75] EST
[23/75] FIN
[24/75] FRA
[25/75] GBR
[26/75] GEO
[27/75] GRC
[28/75] HKG
[29/75] HRV
[30/75] HUN
[31/75] IDN
[32/75] IRL
[33/75] ISL
[34/75] ITA
[35/75] JOR
[36/75] JPN
[

C:\Users\adhiv\AppData\Local\Temp\ipykernel_29616\3188453536.py:1473: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(



Macro columns available:
['Country', 'WB_Code', 'WB_Name', 'WB_Region', 'GDPpcPPP_2018', 'LogGDPpcPPP_2018', 'GNIpcAtlas_2017', 'IncomeGroup_FY2019', 'Gini_Nearest2018', 'Gini_Year', 'PrimaryMacroEligible', 'ExclusionReason', 'GDP_Tertile', 'GDP_Quartile', 'Gini_Tertile', 'IncomeBroad', 'GDP_Gini_Quadrant']

Possible Gini columns: ['Gini_Nearest2018', 'Gini_Year', 'GDP_Gini_Quadrant']


FOCAL PREDICTOR COVERAGE
         Outcome  Variable  N_Systems  MedianMissingRate  MaxMissingRate  MinUnique  MedianWeightedSD  MinWeightedSD
    BEINGBULLIED    BELONG         74             0.0081          0.0429        491            0.9353         0.5984
    BEINGBULLIED  DISCLIMA         74             0.0053          0.0395        308            1.0383         0.8183
    BEINGBULLIED ST004D01T         74             0.0000          0.0000          2            0.4999         0.4973
  BULLY_PHYSICAL    BELONG         75             0.0078          0.0474        490            0.9321         0.5984

In [3]:
%pip install statsmodels

   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ------------- -------------------------- 3.9/11.4 MB 23.4 MB/s eta 0:00:01
   ----------------------------------- ---- 10.0/11.4 MB 24.8 MB/s eta 0:00:01
   ---------------------------------------- 11.4/11.4 MB 23.7 MB/s eta 0:00:00

   ---------------------------------------- 0/6 [wrapt]
   ---------------------------------------- 0/6 [wrapt]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------------- -------------------------- 2/6 [narwhals]
   ------------- -------------------------- 2/6 [narwhals]
   ------------- -------------------------- 2/6 [narwhals]
   ------------- -------------------------- 2/6 [narwhals]
   ------------- -------------------------- 2/6 [narwhals]
   ---


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ============================================================
# PHASE 4B — STAGE-1 SYSTEM-SPECIFIC ASSOCIATION MODELS
# ============================================================
#
# FROZEN STAGE-1 SPECIFICATION
# ----------------------------
#
# Predictors:
#
#   BELONG
#   DISCLIMA
#   ST004D01T
#   AGE
#   ESCS
#
#
# Continuous predictors are standardized WITHIN SYSTEM using
# the survey-weighted mean and SD of the complete-case sample.
#
# Gender remains in its frozen 0/1 coding.
#
#
# OUTCOMES
# --------
#
# BEINGBULLIED:
#   survey-weighted linear regression
#
# Binary subtypes:
#   survey-weighted logistic regression
#
#
# UNCERTAINTY
# -----------
#
# School-clustered sandwich covariance.
#
# We calculate the cluster-robust sandwich explicitly rather
# than relying on statsmodels' cluster covariance with
# frequency weights.
#
#
# IMPORTANT
# ---------
#
# These are adjusted observational associations.
#
# They are NOT:
#   - causal effects
#   - SHAP values
#   - mediation effects
#
# ============================================================


from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd

import statsmodels.api as sm

from scipy.stats import norm


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Frozen Stage-1 specification
# ============================================================

OUTCOMES = [

    "BEINGBULLIED",

    "BULLY_VERBAL",

    "BULLY_THREAT",

    "BULLY_PHYSICAL",

    "BULLY_RELATIONAL",
]


FOCAL_PREDICTORS = [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",
]


COVARIATES = [

    "AGE",

    "ESCS",
]


STAGE1_PREDICTORS = (

    FOCAL_PREDICTORS

    +

    COVARIATES
)


CONTINUOUS_PREDICTORS = [

    "BELONG",

    "DISCLIMA",

    "AGE",

    "ESCS",
]


BINARY_PREDICTORS = [

    "ST004D01T",
]


SCHOOL_COL = "CNTSCHID"

STUDENT_COL = "CNTSTUID"

WEIGHT_COL = "W_FSTUWT"


# ============================================================
# 3. Recover frozen system universe
# ============================================================

ranking_path = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


rankings = pd.read_csv(
    ranking_path
)


SYSTEMS = sorted(
    rankings[
        "Country"
    ]
    .dropna()
    .astype(
        str
    )
    .unique()
)


assert len(
    SYSTEMS
) == 75


print(
    "Systems:",
    len(
        SYSTEMS
    )
)


# ============================================================
# 4. Numeric helper
# ============================================================

def numeric_series(
    series
):

    output = pd.to_numeric(
        series,
        errors="coerce"
    ).astype(
        float
    )


    output[
        ~np.isfinite(
            output
        )
    ] = np.nan


    return output


# ============================================================
# 5. Weighted mean / SD
# ============================================================

def weighted_mean_sd(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    valid = (

        np.isfinite(
            values
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )
    )


    values = values[
        valid
    ]


    weights = weights[
        valid
    ]


    mean = float(
        np.average(
            values,
            weights=weights
        )
    )


    variance = float(
        np.average(
            (
                values
                -
                mean
            )
            **
            2,
            weights=weights
        )
    )


    sd = float(
        np.sqrt(
            variance
        )
    )


    return mean, sd


# ============================================================
# 6. Cluster-robust sandwich covariance
#
# For each school g:
#
#       score_g = X_g' W_g u_g
#
# meat:
#
#       Σ score_g score_g'
#
#
# Small-sample correction:
#
#       G/(G-1) × (N-1)/(N-p)
#
# ============================================================

def cluster_sandwich_linear(
    X,
    residual,
    weights,
    groups
):

    X = np.asarray(
        X,
        dtype=float
    )


    residual = np.asarray(
        residual,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    groups = np.asarray(
        groups
    )


    n, p = X.shape


    unique_groups = np.unique(
        groups
    )


    G = len(
        unique_groups
    )


    # --------------------------------------------------------
    # Bread
    # --------------------------------------------------------

    XtWX = (
        X.T
        @
        (
            weights[
                :,
                None
            ]
            *
            X
        )
    )


    bread = np.linalg.pinv(
        XtWX
    )


    # --------------------------------------------------------
    # Meat
    # --------------------------------------------------------

    meat = np.zeros(
        (
            p,
            p
        ),
        dtype=float
    )


    for group in unique_groups:

        mask = (
            groups
            ==
            group
        )


        Xg = X[
            mask
        ]


        wg = weights[
            mask
        ]


        ug = residual[
            mask
        ]


        score = (
            Xg.T
            @
            (
                wg
                *
                ug
            )
        )


        meat += np.outer(
            score,
            score
        )


    covariance = (
        bread
        @
        meat
        @
        bread
    )


    # --------------------------------------------------------
    # Finite-cluster correction
    # --------------------------------------------------------

    if (
        G
        >
        1
        and
        n
        >
        p
    ):

        correction = (

            G
            /
            (
                G
                -
                1
            )

            *

            (
                n
                -
                1
            )
            /
            (
                n
                -
                p
            )
        )


        covariance *= correction


    return covariance


# ============================================================
# 7. Logistic cluster sandwich
#
# Score:
#
#       w_i x_i (y_i - p_i)
#
# Bread:
#
#       [X' diag(w_i p_i(1-p_i)) X]^-1
# ============================================================

def cluster_sandwich_logistic(
    X,
    y,
    probability,
    weights,
    groups
):

    X = np.asarray(
        X,
        dtype=float
    )


    y = np.asarray(
        y,
        dtype=float
    )


    probability = np.asarray(
        probability,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    groups = np.asarray(
        groups
    )


    n, p = X.shape


    unique_groups = np.unique(
        groups
    )


    G = len(
        unique_groups
    )


    probability = np.clip(
        probability,
        1e-8,
        1
        -
        1e-8
    )


    hessian_weight = (

        weights

        *

        probability

        *

        (
            1
            -
            probability
        )
    )


    information = (

        X.T

        @

        (
            hessian_weight[
                :,
                None
            ]
            *
            X
        )
    )


    bread = np.linalg.pinv(
        information
    )


    meat = np.zeros(
        (
            p,
            p
        ),
        dtype=float
    )


    residual = (
        y
        -
        probability
    )


    for group in unique_groups:

        mask = (
            groups
            ==
            group
        )


        Xg = X[
            mask
        ]


        wg = weights[
            mask
        ]


        rg = residual[
            mask
        ]


        score = (

            Xg.T

            @

            (
                wg
                *
                rg
            )
        )


        meat += np.outer(
            score,
            score
        )


    covariance = (

        bread

        @

        meat

        @

        bread
    )


    if (
        G
        >
        1
        and
        n
        >
        p
    ):

        correction = (

            G
            /
            (
                G
                -
                1
            )

            *

            (
                n
                -
                1
            )
            /
            (
                n
                -
                p
            )
        )


        covariance *= correction


    return covariance


# ============================================================
# 8. Fit one Stage-1 model
# ============================================================

def fit_stage1_model(
    data,
    outcome
):

    y = data[
        outcome
    ].to_numpy(
        dtype=float
    )


    weights = data[
        WEIGHT_COL
    ].to_numpy(
        dtype=float
    )


    groups = data[
        SCHOOL_COL
    ].to_numpy()


    # --------------------------------------------------------
    # Normalize survey weights to mean 1.
    #
    # This does not change coefficient estimates.
    # --------------------------------------------------------

    weights = (
        weights
        /
        np.mean(
            weights
        )
    )


    # --------------------------------------------------------
    # Weighted within-system standardization
    # --------------------------------------------------------

    X_frame = pd.DataFrame(
        index=data.index
    )


    standardization = {}


    for variable in CONTINUOUS_PREDICTORS:

        values = data[
            variable
        ].to_numpy(
            dtype=float
        )


        mean, sd = weighted_mean_sd(
            values,
            weights
        )


        if (
            not np.isfinite(
                sd
            )
            or
            sd
            <=
            0
        ):

            raise RuntimeError(
                f"{variable} has invalid SD."
            )


        X_frame[
            variable
        ] = (
            values
            -
            mean
        ) / sd


        standardization[
            variable
        ] = {

            "mean":
                mean,

            "sd":
                sd,
        }


    # --------------------------------------------------------
    # Frozen 0/1 gender coding
    # --------------------------------------------------------

    X_frame[
        "ST004D01T"
    ] = data[
        "ST004D01T"
    ].to_numpy(
        dtype=float
    )


    # --------------------------------------------------------
    # Preserve frozen predictor order
    # --------------------------------------------------------

    X_frame = X_frame[
        STAGE1_PREDICTORS
    ]


    X_frame = sm.add_constant(
        X_frame,
        has_constant="add"
    )


    coefficient_names = (
        X_frame.columns.tolist()
    )


    X = X_frame.to_numpy(
        dtype=float
    )


    n = len(
        data
    )


    n_schools = len(
        np.unique(
            groups
        )
    )


    # ========================================================
    # Continuous overall bullying
    # ========================================================

    if outcome == "BEINGBULLIED":

        XtWX = (

            X.T

            @

            (
                weights[
                    :,
                    None
                ]
                *
                X
            )
        )


        XtWy = (

            X.T

            @

            (
                weights
                *
                y
            )
        )


        beta = (

            np.linalg.pinv(
                XtWX
            )

            @

            XtWy
        )


        predicted = (
            X
            @
            beta
        )


        residual = (
            y
            -
            predicted
        )


        covariance = (
            cluster_sandwich_linear(
                X=X,
                residual=residual,
                weights=weights,
                groups=groups,
            )
        )


        converged = True


        model_type = (
            "WeightedLinear"
        )


    # ========================================================
    # Binary bullying subtype
    # ========================================================

    else:

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore"
            )


            glm = sm.GLM(

                y,

                X,

                family=sm.families.Binomial(),

                freq_weights=weights,
            )


            fitted = glm.fit(
                maxiter=200,
                tol=1e-9,
                disp=0
            )


        beta = np.asarray(
            fitted.params,
            dtype=float
        )


        probability = np.asarray(
            fitted.predict(
                X
            ),
            dtype=float
        )


        covariance = (
            cluster_sandwich_logistic(
                X=X,
                y=y,
                probability=probability,
                weights=weights,
                groups=groups,
            )
        )


        converged = bool(
            getattr(
                fitted,
                "converged",
                True
            )
        )


        model_type = (
            "WeightedLogistic"
        )


    # ========================================================
    # Standard errors / inference
    # ========================================================

    variance = np.diag(
        covariance
    )


    # Numerical roundoff can create tiny negative values.
    variance = np.where(
        variance
        <
        0,
        np.nan,
        variance
    )


    se = np.sqrt(
        variance
    )


    z_value = (
        beta
        /
        se
    )


    p_value = (
        2
        *
        norm.sf(
            np.abs(
                z_value
            )
        )
    )


    ci_low = (
        beta
        -
        1.96
        *
        se
    )


    ci_high = (
        beta
        +
        1.96
        *
        se
    )


    results = pd.DataFrame(
        {
            "Term":
                coefficient_names,

            "Beta":
                beta,

            "SE":
                se,

            "Variance":
                variance,

            "Z":
                z_value,

            "P":
                p_value,

            "CI95_Low":
                ci_low,

            "CI95_High":
                ci_high,
        }
    )


    return {

        "results":
            results,

        "N":
            n,

        "N_Schools":
            n_schools,

        "Converged":
            converged,

        "ModelType":
            model_type,

        "Standardization":
            standardization,
    }


# ============================================================
# 9. Main Stage-1 fitting loop
# ============================================================

coefficient_rows = []

model_qa_rows = []

standardization_rows = []


total_models = (

    74

    +

    (
        75
        *
        4
    )
)


model_number = 0


for country in SYSTEMS:

    path = (

        PROCESSED_DIR

        /

        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    # --------------------------------------------------------
    # Numeric conversion once
    # --------------------------------------------------------

    numeric = pd.DataFrame(
        index=df.index
    )


    for variable in (

        OUTCOMES

        +

        STAGE1_PREDICTORS

        +

        [
            WEIGHT_COL
        ]
    ):

        numeric[
            variable
        ] = numeric_series(
            df[
                variable
            ]
        )


    numeric[
        SCHOOL_COL
    ] = df[
        SCHOOL_COL
    ]


    numeric[
        STUDENT_COL
    ] = df[
        STUDENT_COL
    ]


    for outcome in OUTCOMES:

        # ----------------------------------------------------
        # Complete cases on Minimal5 + outcome + design vars
        # ----------------------------------------------------

        required = (

            [
                outcome,
                WEIGHT_COL,
                SCHOOL_COL,
            ]

            +

            STAGE1_PREDICTORS
        )


        complete_mask = pd.Series(
            True,
            index=numeric.index
        )


        for variable in (

            [
                outcome,
                WEIGHT_COL,
            ]

            +

            STAGE1_PREDICTORS
        ):

            complete_mask &= (
                numeric[
                    variable
                ]
                .notna()
            )


        complete_mask &= (

            numeric[
                WEIGHT_COL
            ]

            >
            0
        )


        complete_mask &= (

            numeric[
                SCHOOL_COL
            ]
            .notna()
        )


        data = (

            numeric.loc[
                complete_mask,
                required
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Expected KOR overall exclusion
        # ----------------------------------------------------

        if len(
            data
        ) == 0:

            if (
                country
                ==
                "KOR"
                and
                outcome
                ==
                "BEINGBULLIED"
            ):

                continue


            raise RuntimeError(
                f"No complete data for "
                f"{country} {outcome}"
            )


        model_number += 1


        print(
            f"[{model_number:03d}/{total_models}] "
            f"{country} | {outcome}",
            end=""
        )


        # ----------------------------------------------------
        # Binary support QA
        # ----------------------------------------------------

        if outcome != "BEINGBULLIED":

            classes = sorted(
                data[
                    outcome
                ]
                .unique()
            )


            if classes != [
                0.0,
                1.0
            ]:

                raise RuntimeError(
                    f"Unexpected classes for "
                    f"{country} {outcome}: "
                    f"{classes}"
                )


        # ----------------------------------------------------
        # Gender support QA
        # ----------------------------------------------------

        gender_values = sorted(
            data[
                "ST004D01T"
            ]
            .unique()
        )


        if gender_values != [
            0.0,
            1.0
        ]:

            raise RuntimeError(
                f"Unexpected gender coding for "
                f"{country} {outcome}: "
                f"{gender_values}"
            )


        # ----------------------------------------------------
        # Fit
        # ----------------------------------------------------

        try:

            fitted = fit_stage1_model(
                data=data,
                outcome=outcome,
            )


            results = (
                fitted[
                    "results"
                ]
                .copy()
            )


            results.insert(
                0,
                "Outcome",
                outcome
            )


            results.insert(
                0,
                "Country",
                country
            )


            results[
                "ModelType"
            ] = fitted[
                "ModelType"
            ]


            results[
                "N"
            ] = fitted[
                "N"
            ]


            results[
                "N_Schools"
            ] = fitted[
                "N_Schools"
            ]


            coefficient_rows.append(
                results
            )


            # ------------------------------------------------
            # QA
            # ------------------------------------------------

            focal_result = (

                results[
                    results[
                        "Term"
                    ]
                    .isin(
                        FOCAL_PREDICTORS
                    )
                ]
            )


            model_qa_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "ModelType":
                        fitted[
                            "ModelType"
                        ],

                    "N":
                        fitted[
                            "N"
                        ],

                    "N_Schools":
                        fitted[
                            "N_Schools"
                        ],

                    "Converged":
                        fitted[
                            "Converged"
                        ],

                    "AnyNonFiniteBeta":
                        bool(
                            ~np.isfinite(
                                results[
                                    "Beta"
                                ]
                            )
                            .all()
                        ),

                    "AnyNonFiniteSE":
                        bool(
                            ~np.isfinite(
                                results[
                                    "SE"
                                ]
                            )
                            .all()
                        ),

                    "MaxAbsBeta":
                        float(
                            np.nanmax(
                                np.abs(
                                    results[
                                        "Beta"
                                    ]
                                )
                            )
                        ),

                    "MinFocalSE":
                        float(
                            focal_result[
                                "SE"
                            ]
                            .min()
                        ),

                    "MaxFocalSE":
                        float(
                            focal_result[
                                "SE"
                            ]
                            .max()
                        ),
                }
            )


            # ------------------------------------------------
            # Save weighted scaling values
            # ------------------------------------------------

            for variable, values in (
                fitted[
                    "Standardization"
                ]
                .items()
            ):

                standardization_rows.append(
                    {
                        "Country":
                            country,

                        "Outcome":
                            outcome,

                        "Variable":
                            variable,

                        "WeightedMean":
                            values[
                                "mean"
                            ],

                        "WeightedSD":
                            values[
                                "sd"
                            ],
                    }
                )


            print(
                " | N=",
                fitted[
                    "N"
                ],
                "| schools=",
                fitted[
                    "N_Schools"
                ],
                "| converged=",
                fitted[
                    "Converged"
                ]
            )


        except Exception as error:

            model_qa_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "ModelType":
                        (
                            "WeightedLinear"
                            if outcome
                            ==
                            "BEINGBULLIED"
                            else
                            "WeightedLogistic"
                        ),

                    "N":
                        len(
                            data
                        ),

                    "N_Schools":
                        data[
                            SCHOOL_COL
                        ]
                        .nunique(),

                    "Converged":
                        False,

                    "Error":
                        str(
                            error
                        )[
                            :500
                        ],
                }
            )


            print(
                " | ERROR:",
                error
            )


    del (
        df,
        numeric
    )


    gc.collect()


# ============================================================
# 10. Combine outputs
# ============================================================

stage1_coefficients = pd.concat(
    coefficient_rows,
    ignore_index=True
)


stage1_qa = pd.DataFrame(
    model_qa_rows
)


standardization_df = pd.DataFrame(
    standardization_rows
)


# ============================================================
# 11. Save full Stage-1 output
# ============================================================

stage1_coefficients.to_csv(
    OUT_DIR
    / "stage1_system_specific_coefficients.csv",
    index=False
)


stage1_qa.to_csv(
    OUT_DIR
    / "stage1_model_qa.csv",
    index=False
)


standardization_df.to_csv(
    OUT_DIR
    / "stage1_weighted_standardization.csv",
    index=False
)


# ============================================================
# 12. Extract focal coefficients
# ============================================================

focal_coefficients = (

    stage1_coefficients[
        stage1_coefficients[
            "Term"
        ]
        .isin(
            FOCAL_PREDICTORS
        )
    ]
    .copy()
)


focal_coefficients.to_csv(
    OUT_DIR
    / "stage1_focal_coefficients.csv",
    index=False
)


# ============================================================
# 13. Expected coefficient counts
#
# 374 system × outcome profiles
# × 3 focal predictors
# = 1122 focal coefficients
# ============================================================

expected_profiles = 374

expected_focal_rows = (

    expected_profiles

    *

    len(
        FOCAL_PREDICTORS
    )
)


print("\n")
print("=" * 125)
print("STAGE-1 MODEL QA")
print("=" * 125)


print(
    "Expected model profiles:",
    expected_profiles
)


print(
    "Observed model QA rows:",
    len(
        stage1_qa
    )
)


print(
    "Expected focal coefficients:",
    expected_focal_rows
)


print(
    "Observed focal coefficients:",
    len(
        focal_coefficients
    )
)


print(
    "Converged models:",
    int(
        stage1_qa[
            "Converged"
        ]
        .fillna(
            False
        )
        .sum()
    ),
    "/",
    len(
        stage1_qa
    )
)


failed = (

    stage1_qa[
        ~stage1_qa[
            "Converged"
        ]
        .fillna(
            False
        )
    ]
)


print(
    "Failed/nonconverged:",
    len(
        failed
    )
)


if len(
    failed
) > 0:

    print(
        failed
        .to_string(
            index=False
        )
    )


# ============================================================
# 14. Focal coefficient descriptive summary
#
# IMPORTANT:
# This is NOT the formal meta-analysis yet.
# ============================================================

focal_summary = (

    focal_coefficients
    .groupby(
        [
            "Outcome",
            "Term"
        ],
        as_index=False
    )
    .agg(

        N_Systems=(
            "Country",
            "nunique"
        ),

        MeanBeta=(
            "Beta",
            "mean"
        ),

        MedianBeta=(
            "Beta",
            "median"
        ),

        SDBeta=(
            "Beta",
            "std"
        ),

        MinBeta=(
            "Beta",
            "min"
        ),

        MaxBeta=(
            "Beta",
            "max"
        ),

        MeanSE=(
            "SE",
            "mean"
        ),

        MedianSE=(
            "SE",
            "median"
        ),

        PositiveCount=(
            "Beta",
            lambda x:
                int(
                    (
                        x
                        >
                        0
                    )
                    .sum()
                )
        ),

        NegativeCount=(
            "Beta",
            lambda x:
                int(
                    (
                        x
                        <
                        0
                    )
                    .sum()
                )
        ),

        NominalP05Count=(
            "P",
            lambda x:
                int(
                    (
                        x
                        <
                        0.05
                    )
                    .sum()
                )
        ),
    )
)


focal_summary[
    "PositivePct"
] = (

    100

    *

    focal_summary[
        "PositiveCount"
    ]

    /

    focal_summary[
        "N_Systems"
    ]
)


focal_summary[
    "NegativePct"
] = (

    100

    *

    focal_summary[
        "NegativeCount"
    ]

    /

    focal_summary[
        "N_Systems"
    ]
)


focal_summary.to_csv(
    OUT_DIR
    / "stage1_focal_coefficient_descriptive_summary.csv",
    index=False
)


# ============================================================
# 15. Sign-consistency table
# ============================================================

sign_consistency = (

    focal_summary[
        [
            "Outcome",
            "Term",
            "N_Systems",
            "PositiveCount",
            "PositivePct",
            "NegativeCount",
            "NegativePct",
            "MedianBeta",
            "SDBeta",
        ]
    ]
    .copy()
)


# ============================================================
# 16. Largest / smallest slopes
# ============================================================

extreme_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_PREDICTORS:

        temp = (

            focal_coefficients[
                (
                    focal_coefficients[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    focal_coefficients[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .sort_values(
                "Beta"
            )
        )


        if len(
            temp
        ) == 0:

            continue


        for label, subset in [

            (
                "Lowest",
                temp.head(
                    5
                )
            ),

            (
                "Highest",
                temp.tail(
                    5
                )
            ),
        ]:

            for _, row in subset.iterrows():

                extreme_rows.append(
                    {
                        "Outcome":
                            outcome,

                        "Term":
                            term,

                        "Extreme":
                            label,

                        "Country":
                            row[
                                "Country"
                            ],

                        "Beta":
                            row[
                                "Beta"
                            ],

                        "SE":
                            row[
                                "SE"
                            ],

                        "P":
                            row[
                                "P"
                            ],
                    }
                )


extreme_slopes = pd.DataFrame(
    extreme_rows
)


extreme_slopes.to_csv(
    OUT_DIR
    / "stage1_focal_slope_extremes.csv",
    index=False
)


# ============================================================
# 17. Print descriptive Stage-1 results
# ============================================================

print("\n")
print("=" * 125)
print("STAGE-1 FOCAL COEFFICIENT SUMMARY")
print("=" * 125)


print(
    focal_summary
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 18. Sign consistency
# ============================================================

print("\n")
print("=" * 125)
print("STAGE-1 SIGN CONSISTENCY")
print("=" * 125)


print(
    sign_consistency
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 19. Model numerical QA
# ============================================================

print("\n")
print("=" * 125)
print("STAGE-1 NUMERICAL QA")
print("=" * 125)


qa_columns = [

    "AnyNonFiniteBeta",

    "AnyNonFiniteSE",
]


for column in qa_columns:

    if column in stage1_qa.columns:

        print(
            column,
            ":",
            stage1_qa[
                column
            ]
            .value_counts(
                dropna=False
            )
            .to_dict()
        )


if (
    "MaxAbsBeta"
    in stage1_qa.columns
):

    print(
        "\nLargest absolute coefficient in any model:",
        stage1_qa[
            "MaxAbsBeta"
        ]
        .max()
    )


    print(
        "\nProfiles with |beta| > 5:"
    )


    extreme_models = (

        stage1_qa[
            stage1_qa[
                "MaxAbsBeta"
            ]
            >
            5
        ]
    )


    if len(
        extreme_models
    ) == 0:

        print(
            "None."
        )


    else:

        print(
            extreme_models
            .to_string(
                index=False
            )
        )


# ============================================================
# 20. Final status
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 4B STAGE-1 MODELS COMPLETE")
print("=" * 125)


print(
    "\nPrimary Stage-1 specification:"
)


print(
    "Outcome ~ BELONG + DISCLIMA + ST004D01T + AGE + ESCS"
)


print(
    "\nBELONG, DISCLIMA, AGE, and ESCS are "
    "survey-weighted within-system z-scores."
)


print(
    "ST004D01T remains the frozen 0/1 contrast."
)


print(
    "\nNo Stage-2 meta-regression has been fitted yet."
)

Systems: 75
[001/374] ALB | BEINGBULLIED | N= 5814 | schools= 326 | converged= True
[002/374] ALB | BULLY_VERBAL | N= 5908 | schools= 327 | converged= True
[003/374] ALB | BULLY_THREAT | N= 5911 | schools= 326 | converged= True
[004/374] ALB | BULLY_PHYSICAL | N= 5909 | schools= 325 | converged= True
[005/374] ALB | BULLY_RELATIONAL | N= 5912 | schools= 326 | converged= True
[006/374] ARE | BEINGBULLIED | N= 16597 | schools= 750 | converged= True
[007/374] ARE | BULLY_VERBAL | N= 16849 | schools= 752 | converged= True
[008/374] ARE | BULLY_THREAT | N= 16828 | schools= 751 | converged= True
[009/374] ARE | BULLY_PHYSICAL | N= 16823 | schools= 752 | converged= True
[010/374] ARE | BULLY_RELATIONAL | N= 16840 | schools= 752 | converged= True
[011/374] ARG | BEINGBULLIED | N= 7420 | schools= 445 | converged= True
[012/374] ARG | BULLY_VERBAL | N= 7509 | schools= 445 | converged= True
[013/374] ARG | BULLY_THREAT | N= 7478 | schools= 445 | converged= True
[014/374] ARG | BULLY_PHYSICAL | N=

In [5]:
# ============================================================
# PHASE 4C — RANDOM-EFFECTS META-ANALYSIS + META-REGRESSION
# ============================================================
#
# HIERARCHICAL TWO-STAGE DESIGN
# -----------------------------
#
# Stage 1:
#   Already completed.
#
#   For each education system:
#
#       Outcome ~
#           BELONG
#         + DISCLIMA
#         + ST004D01T
#         + AGE
#         + ESCS
#
#   with survey weights and school-clustered SEs.
#
#
# Stage 2:
#
#   Treat each system-specific coefficient as an estimate
#   with known sampling variance.
#
#   Fit:
#
#       1. Random-effects pooled meta-analysis
#
#       2. Random-effects meta-regression using:
#
#           - log GDP per capita PPP
#           - Gini inequality
#           - World Bank region
#
#
# PRIMARY MODERATION FAMILY
# -------------------------
#
# 5 outcomes
# ×
# 3 focal predictors
# ×
# 3 moderators
#
# = 45 pre-specified moderator tests
#
# Benjamini-Hochberg FDR is applied across all 45 tests.
#
#
# IMPORTANT
# ---------
#
# These are observational cross-system associations.
#
# Meta-regression identifies systematic variation in
# coefficients associated with system context.
#
# It does NOT establish causal moderation.
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar

from scipy.stats import (
    norm,
    chi2,
)


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


STAGE1_PATH = (
    OUT_DIR
    / "stage1_focal_coefficients.csv"
)


MACRO_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "macro_validators_with_derived_groups.csv"
)


# ============================================================
# 2. Frozen focal variables
# ============================================================

OUTCOMES = [

    "BEINGBULLIED",

    "BULLY_VERBAL",

    "BULLY_THREAT",

    "BULLY_PHYSICAL",

    "BULLY_RELATIONAL",
]


FOCAL_TERMS = [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",
]


CONTINUOUS_MODERATORS = {

    "LogGDP":
        "LogGDPpcPPP_2018",

    "Gini":
        "Gini_Nearest2018",
}


REGION_MODERATOR = (
    "WB_Region"
)


# ============================================================
# 3. Load Stage-1 coefficients
# ============================================================

stage1 = pd.read_csv(
    STAGE1_PATH
)


macro = pd.read_csv(
    MACRO_PATH
)


# ============================================================
# 4. Basic QA
# ============================================================

expected_stage1_rows = (
    374
    *
    3
)


assert len(
    stage1
) == expected_stage1_rows


assert set(
    stage1[
        "Term"
    ]
    .unique()
) == set(
    FOCAL_TERMS
)


if (
    macro[
        "Country"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Macro table contains duplicate Country rows."
    )


# ============================================================
# 5. Sampling variance
# ============================================================

stage1[
    "SamplingVariance"
] = (
    stage1[
        "SE"
    ]
    **
    2
)


invalid_stage1 = (

    ~np.isfinite(
        stage1[
            "Beta"
        ]
    )

    |

    ~np.isfinite(
        stage1[
            "SamplingVariance"
        ]
    )

    |

    (
        stage1[
            "SamplingVariance"
        ]
        <=
        0
    )
)


if invalid_stage1.any():

    raise RuntimeError(
        "Invalid Stage-1 coefficients or variances found."
    )


# ============================================================
# 6. Merge macro moderators
# ============================================================

moderator_columns = [

    "Country",

    "LogGDPpcPPP_2018",

    "Gini_Nearest2018",

    "WB_Region",
]


for column in moderator_columns:

    if column not in macro.columns:

        raise RuntimeError(
            f"Missing macro column: {column}"
        )


stage2_data = (

    stage1
    .merge(

        macro[
            moderator_columns
        ],

        on="Country",

        how="left",

        validate="many_to_one",
    )
)


stage2_data.to_csv(
    OUT_DIR
    / "stage2_meta_analysis_input.csv",
    index=False
)


# ============================================================
# 7. REML random-effects estimator
# ============================================================

def fit_reml(
    y,
    variance,
    X
):

    y = np.asarray(
        y,
        dtype=float
    )


    variance = np.asarray(
        variance,
        dtype=float
    )


    X = np.asarray(
        X,
        dtype=float
    )


    finite = (

        np.isfinite(
            y
        )

        &

        np.isfinite(
            variance
        )

        &

        (
            variance
            >
            0
        )

        &

        np.isfinite(
            X
        )
        .all(
            axis=1
        )
    )


    y = y[
        finite
    ]


    variance = variance[
        finite
    ]


    X = X[
        finite
    ]


    k = len(
        y
    )


    p = X.shape[
        1
    ]


    if k <= p:

        raise RuntimeError(
            "Not enough systems for meta-regression."
        )


    # --------------------------------------------------------
    # Restricted likelihood objective
    # --------------------------------------------------------

    def reml_objective(
        tau2
    ):

        total_variance = (
            variance
            +
            tau2
        )


        weights = (
            1.0
            /
            total_variance
        )


        XtWX = (

            X.T

            @

            (
                weights[
                    :,
                    None
                ]

                *

                X
            )
        )


        sign, logdet = (
            np.linalg.slogdet(
                XtWX
            )
        )


        if sign <= 0:

            return np.inf


        covariance = (
            np.linalg.pinv(
                XtWX
            )
        )


        beta = (

            covariance

            @

            (
                X.T
                @
                (
                    weights
                    *
                    y
                )
            )
        )


        residual = (
            y
            -
            X
            @
            beta
        )


        weighted_residual_ss = float(

            np.sum(
                weights
                *
                residual
                **
                2
            )
        )


        objective = 0.5 * (

            np.sum(
                np.log(
                    total_variance
                )
            )

            +

            logdet

            +

            weighted_residual_ss
        )


        return float(
            objective
        )


    # --------------------------------------------------------
    # Adaptive upper bound
    # --------------------------------------------------------

    observed_variance = float(
        np.var(
            y,
            ddof=1
        )
    )


    upper = max(

        1.0,

        observed_variance
        *
        10.0,

        float(
            np.median(
                variance
            )
        )
        *
        100.0,
    )


    optimization = minimize_scalar(

        reml_objective,

        bounds=(
            0.0,
            upper
        ),

        method="bounded",

        options={
            "xatol":
                1e-10
        },
    )


    if not optimization.success:

        raise RuntimeError(
            "REML optimization failed."
        )


    tau2_candidate = float(
        optimization.x
    )


    # --------------------------------------------------------
    # Explicitly compare boundary tau² = 0
    # --------------------------------------------------------

    objective_zero = (
        reml_objective(
            0.0
        )
    )


    if (
        objective_zero
        <=
        optimization.fun
    ):

        tau2 = 0.0


    else:

        tau2 = tau2_candidate


    # --------------------------------------------------------
    # Final coefficient fit
    # --------------------------------------------------------

    total_variance = (
        variance
        +
        tau2
    )


    weights = (
        1.0
        /
        total_variance
    )


    XtWX = (

        X.T

        @

        (
            weights[
                :,
                None
            ]

            *

            X
        )
    )


    covariance = (
        np.linalg.pinv(
            XtWX
        )
    )


    beta = (

        covariance

        @

        (
            X.T

            @

            (
                weights
                *
                y
            )
        )
    )


    residual = (

        y

        -

        X
        @
        beta
    )


    return {

        "beta":
            beta,

        "covariance":
            covariance,

        "tau2":
            tau2,

        "weights":
            weights,

        "residual":
            residual,

        "k":
            k,

        "p":
            p,
    }


# ============================================================
# 8. Cochran Q / I² for intercept-only meta-analysis
# ============================================================

def heterogeneity_statistics(
    y,
    variance
):

    y = np.asarray(
        y,
        dtype=float
    )


    variance = np.asarray(
        variance,
        dtype=float
    )


    weights = (
        1.0
        /
        variance
    )


    fixed_mean = float(

        np.sum(
            weights
            *
            y
        )

        /

        np.sum(
            weights
        )
    )


    Q = float(

        np.sum(

            weights

            *

            (
                y
                -
                fixed_mean
            )
            **
            2
        )
    )


    df = (
        len(
            y
        )
        -
        1
    )


    Q_p = float(

        chi2.sf(
            Q,
            df
        )
    )


    if Q > 0:

        I2 = max(

            0.0,

            (
                Q
                -
                df
            )
            /
            Q
        )


    else:

        I2 = 0.0


    return {

        "Q":
            Q,

        "Q_df":
            df,

        "Q_p":
            Q_p,

        "I2":
            (
                I2
                *
                100
            ),
    }


# ============================================================
# 9. Benjamini-Hochberg FDR
# ============================================================

def benjamini_hochberg(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    q_values = np.full(
        len(
            p_values
        ),
        np.nan,
        dtype=float
    )


    finite = np.isfinite(
        p_values
    )


    finite_indices = np.where(
        finite
    )[
        0
    ]


    if len(
        finite_indices
    ) == 0:

        return q_values


    p = p_values[
        finite_indices
    ]


    order = np.argsort(
        p
    )


    ordered_p = p[
        order
    ]


    m = len(
        ordered_p
    )


    ordered_q = (

        ordered_p

        *

        m

        /

        np.arange(
            1,
            m
            +
            1
        )
    )


    ordered_q = np.minimum.accumulate(

        ordered_q[
            ::-1
        ]

    )[
        ::-1
    ]


    ordered_q = np.minimum(
        ordered_q,
        1.0
    )


    original_q = np.empty(
        m,
        dtype=float
    )


    original_q[
        order
    ] = ordered_q


    q_values[
        finite_indices
    ] = original_q


    return q_values


# ============================================================
# 10. Random-effects pooled meta-analysis
# ============================================================

pooled_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp = (

            stage2_data[
                (
                    stage2_data[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage2_data[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .copy()
        )


        y = temp[
            "Beta"
        ].to_numpy(
            dtype=float
        )


        variance = temp[
            "SamplingVariance"
        ].to_numpy(
            dtype=float
        )


        X = np.ones(
            (
                len(
                    temp
                ),
                1
            ),
            dtype=float
        )


        fit = fit_reml(
            y=y,
            variance=variance,
            X=X,
        )


        pooled_beta = float(
            fit[
                "beta"
            ][
                0
            ]
        )


        pooled_se = float(

            np.sqrt(
                fit[
                    "covariance"
                ][
                    0,
                    0
                ]
            )
        )


        ci_low = (
            pooled_beta
            -
            1.96
            *
            pooled_se
        )


        ci_high = (
            pooled_beta
            +
            1.96
            *
            pooled_se
        )


        z_value = (
            pooled_beta
            /
            pooled_se
        )


        p_value = float(

            2

            *

            norm.sf(
                abs(
                    z_value
                )
            )
        )


        heterogeneity = (
            heterogeneity_statistics(
                y=y,
                variance=variance,
            )
        )


        tau2 = float(
            fit[
                "tau2"
            ]
        )


        prediction_se = float(

            np.sqrt(
                tau2
                +
                pooled_se
                **
                2
            )
        )


        prediction_low = (
            pooled_beta
            -
            1.96
            *
            prediction_se
        )


        prediction_high = (
            pooled_beta
            +
            1.96
            *
            prediction_se
        )


        is_logistic = (
            outcome
            !=
            "BEINGBULLIED"
        )


        pooled_rows.append(
            {
                "Outcome":
                    outcome,

                "Term":
                    term,

                "N_Systems":
                    len(
                        temp
                    ),

                "PooledBeta":
                    pooled_beta,

                "SE":
                    pooled_se,

                "CI95_Low":
                    ci_low,

                "CI95_High":
                    ci_high,

                "P":
                    p_value,

                "Tau2":
                    tau2,

                "Tau":
                    np.sqrt(
                        tau2
                    ),

                "Q":
                    heterogeneity[
                        "Q"
                    ],

                "Q_df":
                    heterogeneity[
                        "Q_df"
                    ],

                "Q_p":
                    heterogeneity[
                        "Q_p"
                    ],

                "I2_Pct":
                    heterogeneity[
                        "I2"
                    ],

                "PredictionInterval_Low":
                    prediction_low,

                "PredictionInterval_High":
                    prediction_high,

                "PooledOR":
                    (
                        np.exp(
                            pooled_beta
                        )
                        if is_logistic
                        else np.nan
                    ),

                "OR_CI95_Low":
                    (
                        np.exp(
                            ci_low
                        )
                        if is_logistic
                        else np.nan
                    ),

                "OR_CI95_High":
                    (
                        np.exp(
                            ci_high
                        )
                        if is_logistic
                        else np.nan
                    ),

                "OR_PI_Low":
                    (
                        np.exp(
                            prediction_low
                        )
                        if is_logistic
                        else np.nan
                    ),

                "OR_PI_High":
                    (
                        np.exp(
                            prediction_high
                        )
                        if is_logistic
                        else np.nan
                    ),
            }
        )


pooled_results = pd.DataFrame(
    pooled_rows
)


pooled_results.to_csv(
    OUT_DIR
    / "stage2_random_effects_pooled_results.csv",
    index=False
)


# ============================================================
# 11. Continuous moderator meta-regression
#
# Moderator is standardized across available systems.
#
# Therefore:
#
#   MetaBeta = change in Stage-1 coefficient associated with
#              a 1-SD increase in the system-level moderator.
# ============================================================

moderator_rows = []

moderator_scaling_rows = []


for moderator_name, moderator_column in (
    CONTINUOUS_MODERATORS.items()
):

    for outcome in OUTCOMES:

        for term in FOCAL_TERMS:

            temp = (

                stage2_data[
                    (
                        stage2_data[
                            "Outcome"
                        ]
                        ==
                        outcome
                    )
                    &
                    (
                        stage2_data[
                            "Term"
                        ]
                        ==
                        term
                    )
                ]
                .copy()
            )


            temp = (

                temp[
                    temp[
                        moderator_column
                    ]
                    .notna()
                ]
                .copy()
            )


            moderator = temp[
                moderator_column
            ].to_numpy(
                dtype=float
            )


            moderator_mean = float(
                np.mean(
                    moderator
                )
            )


            moderator_sd = float(
                np.std(
                    moderator,
                    ddof=1
                )
            )


            if (
                not np.isfinite(
                    moderator_sd
                )
                or
                moderator_sd
                <=
                0
            ):

                raise RuntimeError(
                    f"Invalid moderator SD: "
                    f"{moderator_name}"
                )


            moderator_z = (

                moderator

                -

                moderator_mean

            ) / moderator_sd


            y = temp[
                "Beta"
            ].to_numpy(
                dtype=float
            )


            variance = temp[
                "SamplingVariance"
            ].to_numpy(
                dtype=float
            )


            # ------------------------------------------------
            # Null meta-analysis on SAME subset
            # ------------------------------------------------

            X_null = np.ones(
                (
                    len(
                        temp
                    ),
                    1
                )
            )


            null_fit = fit_reml(
                y=y,
                variance=variance,
                X=X_null,
            )


            # ------------------------------------------------
            # Moderator model
            # ------------------------------------------------

            X_mod = np.column_stack(
                [
                    np.ones(
                        len(
                            temp
                        )
                    ),

                    moderator_z,
                ]
            )


            mod_fit = fit_reml(
                y=y,
                variance=variance,
                X=X_mod,
            )


            slope = float(
                mod_fit[
                    "beta"
                ][
                    1
                ]
            )


            slope_se = float(

                np.sqrt(
                    mod_fit[
                        "covariance"
                    ][
                        1,
                        1
                    ]
                )
            )


            z_value = (
                slope
                /
                slope_se
            )


            p_value = float(

                2

                *

                norm.sf(
                    abs(
                        z_value
                    )
                )
            )


            ci_low = (
                slope
                -
                1.96
                *
                slope_se
            )


            ci_high = (
                slope
                +
                1.96
                *
                slope_se
            )


            null_tau2 = float(
                null_fit[
                    "tau2"
                ]
            )


            residual_tau2 = float(
                mod_fit[
                    "tau2"
                ]
            )


            if null_tau2 > 0:

                pseudo_r2 = max(

                    0.0,

                    (
                        null_tau2
                        -
                        residual_tau2
                    )

                    /

                    null_tau2
                )


            else:

                pseudo_r2 = 0.0


            moderator_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Term":
                        term,

                    "Moderator":
                        moderator_name,

                    "ModeratorType":
                        "Continuous",

                    "N_Systems":
                        len(
                            temp
                        ),

                    "ModeratorBeta":
                        slope,

                    "ModeratorSE":
                        slope_se,

                    "CI95_Low":
                        ci_low,

                    "CI95_High":
                        ci_high,

                    "TestStatistic":
                        z_value,

                    "TestDF":
                        1,

                    "P":
                        p_value,

                    "NullTau2":
                        null_tau2,

                    "ResidualTau2":
                        residual_tau2,

                    "PseudoR2_Tau":
                        pseudo_r2,

                    "ReferenceCategory":
                        np.nan,
                }
            )


            moderator_scaling_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Term":
                        term,

                    "Moderator":
                        moderator_name,

                    "RawColumn":
                        moderator_column,

                    "N_Systems":
                        len(
                            temp
                        ),

                    "ModeratorMean":
                        moderator_mean,

                    "ModeratorSD":
                        moderator_sd,
                }
            )


# ============================================================
# 12. Region meta-regression
#
# Omnibus Wald test across K-1 region indicators.
#
# Largest region is automatically used as reference.
# ============================================================

region_detail_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp = (

            stage2_data[
                (
                    stage2_data[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage2_data[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .copy()
        )


        temp = (

            temp[
                temp[
                    REGION_MODERATOR
                ]
                .notna()
            ]
            .copy()
        )


        region_counts = (

            temp[
                REGION_MODERATOR
            ]
            .value_counts()
        )


        # Largest category = reference
        reference_region = (
            region_counts.index[
                0
            ]
        )


        other_regions = [

            region

            for region

            in region_counts.index

            if region
            !=
            reference_region
        ]


        y = temp[
            "Beta"
        ].to_numpy(
            dtype=float
        )


        variance = temp[
            "SamplingVariance"
        ].to_numpy(
            dtype=float
        )


        # ----------------------------------------------------
        # Null model on same subset
        # ----------------------------------------------------

        X_null = np.ones(
            (
                len(
                    temp
                ),
                1
            )
        )


        null_fit = fit_reml(
            y=y,
            variance=variance,
            X=X_null,
        )


        # ----------------------------------------------------
        # Region design matrix
        # ----------------------------------------------------

        dummy_columns = []


        for region in other_regions:

            dummy_columns.append(

                (
                    temp[
                        REGION_MODERATOR
                    ]
                    ==
                    region
                )
                .astype(
                    float
                )
                .to_numpy()
            )


        X_region = np.column_stack(

            [

                np.ones(
                    len(
                        temp
                    )
                )

            ]

            +

            dummy_columns
        )


        region_fit = fit_reml(
            y=y,
            variance=variance,
            X=X_region,
        )


        beta = region_fit[
            "beta"
        ]


        covariance = region_fit[
            "covariance"
        ]


        # ----------------------------------------------------
        # Omnibus Wald test
        # ----------------------------------------------------

        b_region = beta[
            1:
        ]


        covariance_region = covariance[
            1:,
            1:
        ]


        wald = float(

            b_region.T

            @

            np.linalg.pinv(
                covariance_region
            )

            @

            b_region
        )


        df_wald = len(
            b_region
        )


        p_value = float(

            chi2.sf(
                wald,
                df_wald
            )
        )


        null_tau2 = float(
            null_fit[
                "tau2"
            ]
        )


        residual_tau2 = float(
            region_fit[
                "tau2"
            ]
        )


        if null_tau2 > 0:

            pseudo_r2 = max(

                0.0,

                (
                    null_tau2
                    -
                    residual_tau2
                )

                /

                null_tau2
            )


        else:

            pseudo_r2 = 0.0


        moderator_rows.append(
            {
                "Outcome":
                    outcome,

                "Term":
                    term,

                "Moderator":
                    "Region",

                "ModeratorType":
                    "Categorical",

                "N_Systems":
                    len(
                        temp
                    ),

                "ModeratorBeta":
                    np.nan,

                "ModeratorSE":
                    np.nan,

                "CI95_Low":
                    np.nan,

                "CI95_High":
                    np.nan,

                "TestStatistic":
                    wald,

                "TestDF":
                    df_wald,

                "P":
                    p_value,

                "NullTau2":
                    null_tau2,

                "ResidualTau2":
                    residual_tau2,

                "PseudoR2_Tau":
                    pseudo_r2,

                "ReferenceCategory":
                    reference_region,
            }
        )


        # ----------------------------------------------------
        # Region-specific contrasts vs reference
        # ----------------------------------------------------

        for index, region in enumerate(
            other_regions,
            start=1
        ):

            coefficient = float(
                beta[
                    index
                ]
            )


            coefficient_se = float(

                np.sqrt(
                    covariance[
                        index,
                        index
                    ]
                )
            )


            region_detail_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Term":
                        term,

                    "ReferenceRegion":
                        reference_region,

                    "Region":
                        region,

                    "RegionN":
                        int(
                            region_counts[
                                region
                            ]
                        ),

                    "ReferenceN":
                        int(
                            region_counts[
                                reference_region
                            ]
                        ),

                    "BetaDifference":
                        coefficient,

                    "SE":
                        coefficient_se,

                    "CI95_Low":
                        (
                            coefficient
                            -
                            1.96
                            *
                            coefficient_se
                        ),

                    "CI95_High":
                        (
                            coefficient
                            +
                            1.96
                            *
                            coefficient_se
                        ),

                    "P":
                        float(

                            2

                            *

                            norm.sf(

                                abs(
                                    coefficient
                                    /
                                    coefficient_se
                                )
                            )
                        ),
                }
            )


# ============================================================
# 13. Assemble moderator results
# ============================================================

moderator_results = pd.DataFrame(
    moderator_rows
)


assert len(
    moderator_results
) == (
    5
    *
    3
    *
    3
)


assert len(
    moderator_results
) == 45


# ============================================================
# 14. Global FDR across 45 pre-specified moderator tests
# ============================================================

moderator_results[
    "Q_FDR45"
] = benjamini_hochberg(
    moderator_results[
        "P"
    ]
    .to_numpy()
)


moderator_results[
    "FDR05"
] = (
    moderator_results[
        "Q_FDR45"
    ]
    <
    0.05
)


moderator_results[
    "NominalP05"
] = (
    moderator_results[
        "P"
    ]
    <
    0.05
)


# ============================================================
# 15. Save Stage-2 results
# ============================================================

moderator_results.to_csv(
    OUT_DIR
    / "stage2_meta_regression_primary45.csv",
    index=False
)


pd.DataFrame(
    moderator_scaling_rows
).to_csv(
    OUT_DIR
    / "stage2_continuous_moderator_scaling.csv",
    index=False
)


region_detail = pd.DataFrame(
    region_detail_rows
)


region_detail.to_csv(
    OUT_DIR
    / "stage2_region_contrasts.csv",
    index=False
)


# ============================================================
# 16. Moderator coverage / region balance
# ============================================================

coverage_rows = []


for moderator_name, moderator_column in (
    CONTINUOUS_MODERATORS.items()
):

    countries_available = sorted(

        stage2_data.loc[
            stage2_data[
                moderator_column
            ]
            .notna(),
            "Country"
        ]
        .unique()
    )


    countries_missing = sorted(

        stage2_data.loc[
            stage2_data[
                moderator_column
            ]
            .isna(),
            "Country"
        ]
        .unique()
    )


    coverage_rows.append(
        {
            "Moderator":
                moderator_name,

            "N_AvailableSystems":
                len(
                    countries_available
                ),

            "AvailableSystems":
                "|".join(
                    countries_available
                ),

            "MissingSystems":
                "|".join(
                    countries_missing
                ),
        }
    )


region_counts_all = (

    stage2_data[
        [
            "Country",
            REGION_MODERATOR
        ]
    ]
    .drop_duplicates()
    [
        REGION_MODERATOR
    ]
    .value_counts(
        dropna=False
    )
)


coverage_df = pd.DataFrame(
    coverage_rows
)


coverage_df.to_csv(
    OUT_DIR
    / "stage2_moderator_coverage_detail.csv",
    index=False
)


# ============================================================
# 17. Sorted significant / near-significant findings
# ============================================================

moderator_sorted = (

    moderator_results
    .sort_values(
        [
            "Q_FDR45",
            "P"
        ]
    )
    .reset_index(
        drop=True
    )
)


significant_fdr = (

    moderator_sorted[
        moderator_sorted[
            "FDR05"
        ]
    ]
    .copy()
)


nominal_only = (

    moderator_sorted[
        (
            moderator_sorted[
                "NominalP05"
            ]
        )
        &
        (
            ~moderator_sorted[
                "FDR05"
            ]
        )
    ]
    .copy()
)


significant_fdr.to_csv(
    OUT_DIR
    / "stage2_moderation_fdr_significant.csv",
    index=False
)


nominal_only.to_csv(
    OUT_DIR
    / "stage2_moderation_nominal_only.csv",
    index=False
)


# ============================================================
# 18. Pooled result display formatting
# ============================================================

pooled_display = (

    pooled_results[
        [
            "Outcome",
            "Term",
            "N_Systems",
            "PooledBeta",
            "SE",
            "CI95_Low",
            "CI95_High",
            "PooledOR",
            "Tau2",
            "I2_Pct",
            "Q_p",
            "PredictionInterval_Low",
            "PredictionInterval_High",
        ]
    ]
    .copy()
)


# ============================================================
# 19. Print random-effects pooled associations
# ============================================================

print("\n")
print("=" * 130)
print("RANDOM-EFFECTS POOLED STAGE-1 ASSOCIATIONS")
print("=" * 130)


print(
    pooled_display
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print(
    "\nNOTE:"
)


print(
    "For binary subtype outcomes, PooledBeta is a "
    "log-odds coefficient and PooledOR = exp(PooledBeta)."
)


print(
    "For BELONG and DISCLIMA, the coefficient corresponds "
    "to a 1-SD within-system increase."
)


print(
    "For ST004D01T, the coefficient is the frozen "
    "1-vs-0 gender contrast."
)


# ============================================================
# 20. Heterogeneity summary
# ============================================================

print("\n")
print("=" * 130)
print("CROSS-SYSTEM HETEROGENEITY")
print("=" * 130)


print(
    pooled_results[
        [
            "Outcome",
            "Term",
            "N_Systems",
            "Tau2",
            "I2_Pct",
            "Q",
            "Q_df",
            "Q_p",
            "PredictionInterval_Low",
            "PredictionInterval_High",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Region balance
# ============================================================

print("\n")
print("=" * 130)
print("REGION BALANCE")
print("=" * 130)


print(
    region_counts_all
    .to_string()
)


print(
    "\nInterpret region results cautiously where categories "
    "contain very few systems."
)


# ============================================================
# 22. All primary moderator tests
# ============================================================

print("\n")
print("=" * 130)
print("PRIMARY META-REGRESSION MODERATOR TESTS — 45 TEST FAMILY")
print("=" * 130)


print(
    moderator_sorted[
        [
            "Outcome",
            "Term",
            "Moderator",
            "N_Systems",
            "ModeratorBeta",
            "ModeratorSE",
            "TestStatistic",
            "TestDF",
            "P",
            "Q_FDR45",
            "NullTau2",
            "ResidualTau2",
            "PseudoR2_Tau",
            "ReferenceCategory",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 23. FDR-significant moderation
# ============================================================

print("\n")
print("=" * 130)
print("FDR-SIGNIFICANT MODERATION")
print("=" * 130)


print(
    "Number significant after BH-FDR across 45 tests:",
    len(
        significant_fdr
    )
)


if len(
    significant_fdr
) == 0:

    print(
        "None."
    )


else:

    print(
        significant_fdr[
            [
                "Outcome",
                "Term",
                "Moderator",
                "N_Systems",
                "ModeratorBeta",
                "ModeratorSE",
                "P",
                "Q_FDR45",
                "PseudoR2_Tau",
                "ReferenceCategory",
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 24. Nominal-only moderation
# ============================================================

print("\n")
print("=" * 130)
print("NOMINAL P < .05 BUT NOT FDR-SIGNIFICANT")
print("=" * 130)


if len(
    nominal_only
) == 0:

    print(
        "None."
    )


else:

    print(
        nominal_only[
            [
                "Outcome",
                "Term",
                "Moderator",
                "N_Systems",
                "ModeratorBeta",
                "ModeratorSE",
                "P",
                "Q_FDR45",
                "PseudoR2_Tau",
                "ReferenceCategory",
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 25. Region contrasts
#
# These are descriptive follow-ups.
#
# The PRIMARY region inference is the omnibus FDR-adjusted
# Region test above.
# ============================================================

print("\n")
print("=" * 130)
print("REGION CONTRASTS — DESCRIPTIVE FOLLOW-UP")
print("=" * 130)


print(
    region_detail
    .sort_values(
        [
            "Outcome",
            "Term",
            "P"
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 26. Final QA / status
# ============================================================

print("\n")
print("=" * 130)
print("PHASE 4C RANDOM-EFFECTS META-ANALYSIS COMPLETE")
print("=" * 130)


print(
    "Pooled effect profiles:",
    len(
        pooled_results
    )
)


print(
    "Expected pooled profiles:",
    5
    *
    3
)


print(
    "Primary moderator tests:",
    len(
        moderator_results
    )
)


print(
    "Expected primary moderator tests:",
    45
)


print(
    "FDR-significant moderator tests:",
    len(
        significant_fdr
    )
)


print(
    "\nNo mediation analysis has been performed."
)



RANDOM-EFFECTS POOLED STAGE-1 ASSOCIATIONS
         Outcome      Term  N_Systems  PooledBeta      SE  CI95_Low  CI95_High  PooledOR    Tau2   I2_Pct  Q_p  PredictionInterval_Low  PredictionInterval_High
    BEINGBULLIED    BELONG         74    -0.26559 0.00646  -0.27826   -0.25293       NaN 0.00281 91.97157  0.0                -0.37029                 -0.16090
    BEINGBULLIED  DISCLIMA         74    -0.14143 0.00373  -0.14873   -0.13412       NaN 0.00075 75.82679  0.0                -0.19543                 -0.08742
    BEINGBULLIED ST004D01T         74     0.24164 0.01248   0.21719    0.26610       NaN 0.01045 92.26527  0.0                 0.03978                  0.44350
    BULLY_VERBAL    BELONG         75    -0.60726 0.02212  -0.65063   -0.56390   0.54484 0.03243 92.83008  0.0                -0.96289                 -0.25164
    BULLY_VERBAL  DISCLIMA         75    -0.26270 0.00916  -0.28064   -0.24475   0.76898 0.00416 72.41757  0.0                -0.39037                 -0.1

In [6]:
# ============================================================
# PHASE 4D — JOINT MODERATION + ROBUSTNESS
# ============================================================
#
# PURPOSE
# -------
# Phase 4C tested GDP, Gini, and region separately.
#
# Here we ask the more rigorous questions:
#
#   1. Does GDP moderate Stage-1 coefficients AFTER
#      accounting for region?
#
#   2. Does region moderate Stage-1 coefficients AFTER
#      accounting for GDP?
#
#   3. Does Gini add information BEYOND GDP + region?
#
#   4. Are GDP/region conclusions robust when the very small
#      North America group (n=2) is excluded?
#
#
# INFERENCE
# ---------
# Random-effects REML meta-regression +
# modified Knapp-Hartung inference.
#
# The modification uses:
#
#     scale = max(Q_resid / df_resid, 1)
#
# so the KH adjustment cannot make the conventional
# inverse-variance SE smaller.
#
#
# FDR FAMILIES
# ------------
#
# Primary joint moderation:
#   5 outcomes × 3 terms × 2 tests
#   = 30 tests
#
#       GDP adjusted for region
#       Region adjusted for GDP
#
# Gini incremental sensitivity:
#   5 outcomes × 3 terms
#   = 15 tests
#
# North-America-excluded sensitivity:
#   30 tests
#
#
# OBSERVATIONAL ASSOCIATIONS ONLY.
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    t as student_t,
    f as f_dist,
)


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


INPUT_PATH = (
    OUT_DIR
    / "stage2_meta_analysis_input.csv"
)


# ============================================================
# 2. Verify Phase 4C functions are available
# ============================================================

required_functions = [
    "fit_reml",
]


for function_name in required_functions:

    if function_name not in globals():

        raise RuntimeError(
            f"{function_name} is not available. "
            "Run Phase 4C first."
        )


# ============================================================
# 3. Variables
# ============================================================

OUTCOMES = [

    "BEINGBULLIED",

    "BULLY_VERBAL",

    "BULLY_THREAT",

    "BULLY_PHYSICAL",

    "BULLY_RELATIONAL",
]


FOCAL_TERMS = [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",
]


GDP_COL = (
    "LogGDPpcPPP_2018"
)


GINI_COL = (
    "Gini_Nearest2018"
)


REGION_COL = (
    "WB_Region"
)


# ============================================================
# 4. Load Stage-2 input
# ============================================================

stage2 = pd.read_csv(
    INPUT_PATH
)


stage2[
    "SamplingVariance"
] = (

    stage2[
        "SE"
    ]

    **

    2
)


# ============================================================
# 5. Benjamini-Hochberg
# ============================================================

def bh_fdr(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    output = np.full(
        len(
            p_values
        ),
        np.nan
    )


    valid = np.isfinite(
        p_values
    )


    indices = np.where(
        valid
    )[
        0
    ]


    if len(
        indices
    ) == 0:

        return output


    p = p_values[
        indices
    ]


    order = np.argsort(
        p
    )


    ordered = p[
        order
    ]


    m = len(
        ordered
    )


    q = (

        ordered

        *

        m

        /

        np.arange(
            1,
            m
            +
            1
        )
    )


    q = np.minimum.accumulate(
        q[
            ::-1
        ]
    )[
        ::-1
    ]


    q = np.clip(
        q,
        0,
        1
    )


    restored = np.empty(
        m
    )


    restored[
        order
    ] = q


    output[
        indices
    ] = restored


    return output


# ============================================================
# 6. Modified Knapp-Hartung covariance
# ============================================================

def modified_knapp_hartung(
    fit
):

    weights = np.asarray(
        fit[
            "weights"
        ],
        dtype=float
    )


    residual = np.asarray(
        fit[
            "residual"
        ],
        dtype=float
    )


    k = int(
        fit[
            "k"
        ]
    )


    p = int(
        fit[
            "p"
        ]
    )


    df_residual = (
        k
        -
        p
    )


    if df_residual <= 0:

        raise RuntimeError(
            "Invalid meta-regression residual degrees "
            "of freedom."
        )


    Q_residual = float(

        np.sum(

            weights

            *

            residual
            **
            2
        )
    )


    kh_raw_scale = (

        Q_residual

        /

        df_residual
    )


    # Conservative modified KH
    kh_scale = max(
        1.0,
        kh_raw_scale
    )


    covariance = (

        fit[
            "covariance"
        ]

        *

        kh_scale
    )


    return {

        "covariance":
            covariance,

        "df":
            df_residual,

        "Q_residual":
            Q_residual,

        "KH_raw_scale":
            kh_raw_scale,

        "KH_scale":
            kh_scale,
    }


# ============================================================
# 7. Standardize continuous moderator
# ============================================================

def zscore(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    mean = float(
        np.mean(
            values
        )
    )


    sd = float(
        np.std(
            values,
            ddof=1
        )
    )


    if (
        not np.isfinite(
            sd
        )
        or
        sd
        <=
        0
    ):

        raise RuntimeError(
            "Invalid moderator SD."
        )


    standardized = (

        values

        -

        mean

    ) / sd


    return (
        standardized,
        mean,
        sd,
    )


# ============================================================
# 8. Region dummy matrix
# ============================================================

def region_design(
    region_series
):

    region_series = pd.Series(
        region_series
    )


    counts = (
        region_series
        .value_counts()
    )


    # Largest region is reference
    reference = (
        counts.index[
            0
        ]
    )


    others = [

        region

        for region
        in counts.index

        if region
        !=
        reference
    ]


    dummy_matrix = np.column_stack(
        [

            (
                region_series
                ==
                region
            )
            .astype(
                float
            )
            .to_numpy()

            for region
            in others
        ]
    )


    return (
        reference,
        others,
        dummy_matrix,
        counts,
    )


# ============================================================
# 9. Fit primary GDP + Region model
# ============================================================

def fit_gdp_region_model(
    temp
):

    temp = (

        temp
        .dropna(
            subset=[
                GDP_COL,
                REGION_COL,
                "Beta",
                "SamplingVariance",
            ]
        )
        .copy()
    )


    gdp_z, gdp_mean, gdp_sd = (
        zscore(
            temp[
                GDP_COL
            ]
        )
    )


    (
        reference,
        other_regions,
        region_matrix,
        region_counts,
    ) = region_design(
        temp[
            REGION_COL
        ]
    )


    X = np.column_stack(
        [

            np.ones(
                len(
                    temp
                )
            ),

            gdp_z,

            region_matrix,
        ]
    )


    names = (

        [
            "Intercept",
            "LogGDP_z",
        ]

        +

        [
            f"Region::{region}"
            for region
            in other_regions
        ]
    )


    fit = fit_reml(

        y=temp[
            "Beta"
        ].to_numpy(
            dtype=float
        ),

        variance=temp[
            "SamplingVariance"
        ].to_numpy(
            dtype=float
        ),

        X=X,
    )


    kh = modified_knapp_hartung(
        fit
    )


    # Intercept-only model on exact same systems
    null_fit = fit_reml(

        y=temp[
            "Beta"
        ].to_numpy(
            dtype=float
        ),

        variance=temp[
            "SamplingVariance"
        ].to_numpy(
            dtype=float
        ),

        X=np.ones(
            (
                len(
                    temp
                ),
                1
            )
        ),
    )


    return {

        "data":
            temp,

        "fit":
            fit,

        "kh":
            kh,

        "names":
            names,

        "reference":
            reference,

        "other_regions":
            other_regions,

        "region_counts":
            region_counts,

        "GDP_mean":
            gdp_mean,

        "GDP_sd":
            gdp_sd,

        "null_tau2":
            float(
                null_fit[
                    "tau2"
                ]
            ),
    }


# ============================================================
# 10. Extract GDP + Region tests
# ============================================================

def test_gdp_region(
    model
):

    fit = model[
        "fit"
    ]


    kh = model[
        "kh"
    ]


    beta = np.asarray(
        fit[
            "beta"
        ]
    )


    covariance = np.asarray(
        kh[
            "covariance"
        ]
    )


    df = int(
        kh[
            "df"
        ]
    )


    # --------------------------------------------------------
    # GDP adjusted for region
    # --------------------------------------------------------

    estimate = float(
        beta[
            1
        ]
    )


    se = float(

        np.sqrt(
            covariance[
                1,
                1
            ]
        )
    )


    statistic = (
        estimate
        /
        se
    )


    p_gdp = float(

        2

        *

        student_t.sf(
            abs(
                statistic
            ),
            df
        )
    )


    critical = float(
        student_t.ppf(
            0.975,
            df
        )
    )


    gdp_result = {

        "Estimate":
            estimate,

        "SE":
            se,

        "Statistic":
            statistic,

        "DF1":
            1,

        "DF2":
            df,

        "P":
            p_gdp,

        "CI95_Low":
            estimate
            -
            critical
            *
            se,

        "CI95_High":
            estimate
            +
            critical
            *
            se,
    }


    # --------------------------------------------------------
    # Region adjusted for GDP
    # --------------------------------------------------------

    region_beta = (
        beta[
            2:
        ]
    )


    region_cov = (
        covariance[
            2:,
            2:
        ]
    )


    r = len(
        region_beta
    )


    if r == 0:

        raise RuntimeError(
            "No region contrasts available."
        )


    wald = float(

        region_beta.T

        @

        np.linalg.pinv(
            region_cov
        )

        @

        region_beta
    )


    F_stat = (

        wald

        /

        r
    )


    p_region = float(

        f_dist.sf(
            F_stat,
            r,
            df
        )
    )


    region_result = {

        "Estimate":
            np.nan,

        "SE":
            np.nan,

        "Statistic":
            F_stat,

        "DF1":
            r,

        "DF2":
            df,

        "P":
            p_region,

        "CI95_Low":
            np.nan,

        "CI95_High":
            np.nan,
    }


    return (
        gdp_result,
        region_result,
    )


# ============================================================
# 11. PRIMARY JOINT MODEL:
# GDP + Region
# ============================================================

primary_rows = []

primary_coefficient_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp = (

            stage2[
                (
                    stage2[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage2[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .copy()
        )


        model = fit_gdp_region_model(
            temp
        )


        (
            gdp_test,
            region_test,
        ) = test_gdp_region(
            model
        )


        fit = model[
            "fit"
        ]


        kh = model[
            "kh"
        ]


        null_tau2 = model[
            "null_tau2"
        ]


        residual_tau2 = float(
            fit[
                "tau2"
            ]
        )


        pseudo_r2 = (

            max(
                0.0,
                (
                    null_tau2
                    -
                    residual_tau2
                )
                /
                null_tau2
            )

            if null_tau2
            >
            0

            else 0.0
        )


        # ----------------------------------------------------
        # GDP adjusted for region
        # ----------------------------------------------------

        primary_rows.append(
            {

                "Outcome":
                    outcome,

                "Term":
                    term,

                "Test":
                    "LogGDP_adjusted_for_Region",

                "N_Systems":
                    len(
                        model[
                            "data"
                        ]
                    ),

                **gdp_test,

                "Tau2":
                    residual_tau2,

                "NullTau2":
                    null_tau2,

                "PseudoR2_Tau":
                    pseudo_r2,

                "KH_Scale":
                    kh[
                        "KH_scale"
                    ],

                "ReferenceRegion":
                    model[
                        "reference"
                    ],
            }
        )


        # ----------------------------------------------------
        # Region adjusted for GDP
        # ----------------------------------------------------

        primary_rows.append(
            {

                "Outcome":
                    outcome,

                "Term":
                    term,

                "Test":
                    "Region_adjusted_for_LogGDP",

                "N_Systems":
                    len(
                        model[
                            "data"
                        ]
                    ),

                **region_test,

                "Tau2":
                    residual_tau2,

                "NullTau2":
                    null_tau2,

                "PseudoR2_Tau":
                    pseudo_r2,

                "KH_Scale":
                    kh[
                        "KH_scale"
                    ],

                "ReferenceRegion":
                    model[
                        "reference"
                    ],
            }
        )


        # ----------------------------------------------------
        # Save all joint-model coefficients
        # ----------------------------------------------------

        covariance = (
            kh[
                "covariance"
            ]
        )


        df = (
            kh[
                "df"
            ]
        )


        critical = float(
            student_t.ppf(
                0.975,
                df
            )
        )


        for index, name in enumerate(
            model[
                "names"
            ]
        ):

            coefficient = float(
                fit[
                    "beta"
                ][
                    index
                ]
            )


            se = float(
                np.sqrt(
                    covariance[
                        index,
                        index
                    ]
                )
            )


            primary_coefficient_rows.append(
                {

                    "Outcome":
                        outcome,

                    "Term":
                        term,

                    "MetaTerm":
                        name,

                    "Estimate":
                        coefficient,

                    "SE":
                        se,

                    "DF":
                        df,

                    "CI95_Low":
                        coefficient
                        -
                        critical
                        *
                        se,

                    "CI95_High":
                        coefficient
                        +
                        critical
                        *
                        se,

                    "P":
                        float(
                            2
                            *
                            student_t.sf(
                                abs(
                                    coefficient
                                    /
                                    se
                                ),
                                df
                            )
                        ),

                    "ReferenceRegion":
                        model[
                            "reference"
                        ],
                }
            )


primary_results = pd.DataFrame(
    primary_rows
)


assert len(
    primary_results
) == 30


primary_results[
    "Q_FDR30"
] = bh_fdr(
    primary_results[
        "P"
    ]
)


primary_results[
    "FDR05"
] = (
    primary_results[
        "Q_FDR30"
    ]
    <
    0.05
)


primary_results.to_csv(
    OUT_DIR
    / "stage2_joint_gdp_region_primary30.csv",
    index=False
)


pd.DataFrame(
    primary_coefficient_rows
).to_csv(
    OUT_DIR
    / "stage2_joint_gdp_region_coefficients.csv",
    index=False
)


# ============================================================
# 12. GINI INCREMENTAL MODEL
#
# Full model:
#
#     GDP + Gini + Region
#
# Test:
#
#     Gini | GDP + Region
#
# ============================================================

gini_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp = (

            stage2[
                (
                    stage2[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage2[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .dropna(
                subset=[
                    GDP_COL,
                    GINI_COL,
                    REGION_COL,
                    "Beta",
                    "SamplingVariance",
                ]
            )
            .copy()
        )


        gdp_z, _, _ = zscore(
            temp[
                GDP_COL
            ]
        )


        gini_z, gini_mean, gini_sd = (
            zscore(
                temp[
                    GINI_COL
                ]
            )
        )


        (
            reference,
            other_regions,
            region_matrix,
            region_counts,
        ) = region_design(
            temp[
                REGION_COL
            ]
        )


        # ----------------------------------------------------
        # Reduced:
        # GDP + region
        # ----------------------------------------------------

        X_reduced = np.column_stack(
            [

                np.ones(
                    len(
                        temp
                    )
                ),

                gdp_z,

                region_matrix,
            ]
        )


        reduced_fit = fit_reml(

            y=temp[
                "Beta"
            ],

            variance=temp[
                "SamplingVariance"
            ],

            X=X_reduced,
        )


        # ----------------------------------------------------
        # Full:
        # GDP + Gini + region
        # ----------------------------------------------------

        X_full = np.column_stack(
            [

                np.ones(
                    len(
                        temp
                    )
                ),

                gdp_z,

                gini_z,

                region_matrix,
            ]
        )


        full_fit = fit_reml(

            y=temp[
                "Beta"
            ],

            variance=temp[
                "SamplingVariance"
            ],

            X=X_full,
        )


        kh = modified_knapp_hartung(
            full_fit
        )


        covariance = (
            kh[
                "covariance"
            ]
        )


        df = int(
            kh[
                "df"
            ]
        )


        # Gini = column 2
        estimate = float(
            full_fit[
                "beta"
            ][
                2
            ]
        )


        se = float(
            np.sqrt(
                covariance[
                    2,
                    2
                ]
            )
        )


        statistic = (
            estimate
            /
            se
        )


        p_value = float(

            2

            *

            student_t.sf(
                abs(
                    statistic
                ),
                df
            )
        )


        critical = float(
            student_t.ppf(
                0.975,
                df
            )
        )


        reduced_tau2 = float(
            reduced_fit[
                "tau2"
            ]
        )


        full_tau2 = float(
            full_fit[
                "tau2"
            ]
        )


        incremental_r2 = (

            max(
                0.0,

                (
                    reduced_tau2
                    -
                    full_tau2
                )

                /

                reduced_tau2
            )

            if reduced_tau2
            >
            0

            else 0.0
        )


        gini_rows.append(
            {

                "Outcome":
                    outcome,

                "Term":
                    term,

                "N_Systems":
                    len(
                        temp
                    ),

                "GiniBeta":
                    estimate,

                "SE":
                    se,

                "T":
                    statistic,

                "DF":
                    df,

                "P":
                    p_value,

                "CI95_Low":
                    estimate
                    -
                    critical
                    *
                    se,

                "CI95_High":
                    estimate
                    +
                    critical
                    *
                    se,

                "ReducedTau2":
                    reduced_tau2,

                "FullTau2":
                    full_tau2,

                "IncrementalPseudoR2":
                    incremental_r2,

                "KH_Scale":
                    kh[
                        "KH_scale"
                    ],

                "ReferenceRegion":
                    reference,

                "GiniMean":
                    gini_mean,

                "GiniSD":
                    gini_sd,
            }
        )


gini_results = pd.DataFrame(
    gini_rows
)


assert len(
    gini_results
) == 15


gini_results[
    "Q_FDR15"
] = bh_fdr(
    gini_results[
        "P"
    ]
)


gini_results[
    "FDR05"
] = (
    gini_results[
        "Q_FDR15"
    ]
    <
    0.05
)


gini_results.to_csv(
    OUT_DIR
    / "stage2_gini_incremental_after_gdp_region.csv",
    index=False
)


# ============================================================
# 13. REGION SENSITIVITY:
# remove North America (n=2)
# ============================================================

sensitivity_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp = (

            stage2[
                (
                    stage2[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage2[
                        "Term"
                    ]
                    ==
                    term
                )
                &
                (
                    stage2[
                        REGION_COL
                    ]
                    !=
                    "North America"
                )
            ]
            .copy()
        )


        model = fit_gdp_region_model(
            temp
        )


        (
            gdp_test,
            region_test,
        ) = test_gdp_region(
            model
        )


        fit = model[
            "fit"
        ]


        kh = model[
            "kh"
        ]


        # GDP
        sensitivity_rows.append(
            {

                "Outcome":
                    outcome,

                "Term":
                    term,

                "Test":
                    "LogGDP_adjusted_for_Region",

                "N_Systems":
                    len(
                        model[
                            "data"
                        ]
                    ),

                **gdp_test,

                "Tau2":
                    fit[
                        "tau2"
                    ],

                "KH_Scale":
                    kh[
                        "KH_scale"
                    ],

                "ReferenceRegion":
                    model[
                        "reference"
                    ],
            }
        )


        # Region
        sensitivity_rows.append(
            {

                "Outcome":
                    outcome,

                "Term":
                    term,

                "Test":
                    "Region_adjusted_for_LogGDP",

                "N_Systems":
                    len(
                        model[
                            "data"
                        ]
                    ),

                **region_test,

                "Tau2":
                    fit[
                        "tau2"
                    ],

                "KH_Scale":
                    kh[
                        "KH_scale"
                    ],

                "ReferenceRegion":
                    model[
                        "reference"
                    ],
            }
        )


sensitivity_results = pd.DataFrame(
    sensitivity_rows
)


assert len(
    sensitivity_results
) == 30


sensitivity_results[
    "Q_FDR30"
] = bh_fdr(
    sensitivity_results[
        "P"
    ]
)


sensitivity_results[
    "FDR05"
] = (
    sensitivity_results[
        "Q_FDR30"
    ]
    <
    0.05
)


sensitivity_results.to_csv(
    OUT_DIR
    / "stage2_joint_gdp_region_no_north_america.csv",
    index=False
)


# ============================================================
# 14. Compare primary and sensitivity conclusions
# ============================================================

robustness = (

    primary_results[
        [
            "Outcome",
            "Term",
            "Test",
            "P",
            "Q_FDR30",
            "FDR05",
        ]
    ]
    .rename(
        columns={

            "P":
                "PrimaryP",

            "Q_FDR30":
                "PrimaryQ",

            "FDR05":
                "PrimaryFDR05",
        }
    )
    .merge(

        sensitivity_results[
            [
                "Outcome",
                "Term",
                "Test",
                "P",
                "Q_FDR30",
                "FDR05",
            ]
        ]
        .rename(
            columns={

                "P":
                    "NoNorthAmericaP",

                "Q_FDR30":
                    "NoNorthAmericaQ",

                "FDR05":
                    "NoNorthAmericaFDR05",
            }
        ),

        on=[
            "Outcome",
            "Term",
            "Test",
        ],

        how="inner",

        validate="one_to_one",
    )
)


robustness[
    "SameFDRConclusion"
] = (

    robustness[
        "PrimaryFDR05"
    ]

    ==

    robustness[
        "NoNorthAmericaFDR05"
    ]
)


robustness.to_csv(
    OUT_DIR
    / "stage2_gdp_region_north_america_robustness.csv",
    index=False
)


# ============================================================
# 15. Print primary adjusted moderation
# ============================================================

print("\n")
print("=" * 130)
print("PRIMARY JOINT MODERATION: GDP + REGION")
print("=" * 130)


print(
    primary_results
    .sort_values(
        [
            "Q_FDR30",
            "P"
        ]
    )
    [
        [
            "Outcome",
            "Term",
            "Test",
            "N_Systems",
            "Estimate",
            "SE",
            "Statistic",
            "DF1",
            "DF2",
            "P",
            "Q_FDR30",
            "PseudoR2_Tau",
            "KH_Scale",
            "ReferenceRegion",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 16. Significant primary joint moderation
# ============================================================

print("\n")
print("=" * 130)
print("FDR-SIGNIFICANT JOINT GDP/REGION MODERATION")
print("=" * 130)


primary_significant = (

    primary_results[
        primary_results[
            "FDR05"
        ]
    ]
    .sort_values(
        "Q_FDR30"
    )
)


print(
    "Significant:",
    len(
        primary_significant
    ),
    "/ 30"
)


if len(
    primary_significant
) > 0:

    print(
        primary_significant[
            [
                "Outcome",
                "Term",
                "Test",
                "Estimate",
                "SE",
                "P",
                "Q_FDR30",
                "PseudoR2_Tau",
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 17. Gini incremental results
# ============================================================

print("\n")
print("=" * 130)
print("GINI INCREMENTAL MODERATION AFTER GDP + REGION")
print("=" * 130)


print(
    gini_results
    .sort_values(
        [
            "Q_FDR15",
            "P"
        ]
    )
    [
        [
            "Outcome",
            "Term",
            "N_Systems",
            "GiniBeta",
            "SE",
            "T",
            "DF",
            "P",
            "Q_FDR15",
            "IncrementalPseudoR2",
            "KH_Scale",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print(
    "\nGini FDR-significant after GDP + region:",
    int(
        gini_results[
            "FDR05"
        ]
        .sum()
    ),
    "/ 15"
)


# ============================================================
# 18. North America exclusion sensitivity
# ============================================================

print("\n")
print("=" * 130)
print("NORTH AMERICA EXCLUSION SENSITIVITY")
print("=" * 130)


print(
    "Same FDR conclusion:",
    int(
        robustness[
            "SameFDRConclusion"
        ]
        .sum()
    ),
    "/",
    len(
        robustness
    )
)


changed = (

    robustness[
        ~robustness[
            "SameFDRConclusion"
        ]
    ]
)


if len(
    changed
) == 0:

    print(
        "No FDR conclusions changed."
    )


else:

    print(
        changed
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 19. Numerical / design QA
# ============================================================

print("\n")
print("=" * 130)
print("PHASE 4D QA")
print("=" * 130)


print(
    "Primary tests:",
    len(
        primary_results
    ),
    "(expected 30)"
)


print(
    "Gini incremental tests:",
    len(
        gini_results
    ),
    "(expected 15)"
)


print(
    "North-America-excluded sensitivity tests:",
    len(
        sensitivity_results
    ),
    "(expected 30)"
)


print(
    "Primary FDR significant:",
    int(
        primary_results[
            "FDR05"
        ]
        .sum()
    )
)


print(
    "Gini incremental FDR significant:",
    int(
        gini_results[
            "FDR05"
        ]
        .sum()
    )
)


print("\n")
print("=" * 130)
print("PHASE 4D JOINT MODERATION ROBUSTNESS COMPLETE")
print("=" * 130)


print(
    "\nDo not freeze moderation conclusions until these "
    "joint-adjusted results are inspected."
)



PRIMARY JOINT MODERATION: GDP + REGION
         Outcome      Term                       Test  N_Systems  Estimate      SE  Statistic  DF1  DF2       P  Q_FDR30  PseudoR2_Tau  KH_Scale       ReferenceRegion
    BEINGBULLIED ST004D01T Region_adjusted_for_LogGDP         71       NaN     NaN   12.33149    4   65 0.00000  0.00000       0.42520   1.00000 Europe & Central Asia
BULLY_RELATIONAL ST004D01T Region_adjusted_for_LogGDP         72       NaN     NaN   10.63862    4   66 0.00000  0.00002       0.41284   1.00000 Europe & Central Asia
  BULLY_PHYSICAL ST004D01T LogGDP_adjusted_for_Region         72   0.10837 0.02236    4.84725    1   66 0.00001  0.00008       0.42910   1.00000 Europe & Central Asia
BULLY_RELATIONAL    BELONG LogGDP_adjusted_for_Region         72  -0.08620 0.01846   -4.67043    1   66 0.00002  0.00010       0.34570   1.00000 Europe & Central Asia
    BULLY_THREAT  DISCLIMA LogGDP_adjusted_for_Region         72  -0.04931 0.01061   -4.64855    1   66 0.00002  0.00010    

In [7]:
# ============================================================
# PHASE 4E — MODERATION INTERPRETATION + FREEZE TABLES
# ============================================================
#
# PURPOSE
# -------
# NO NEW HYPOTHESIS TESTS.
#
# Convert the already-fitted GDP + region meta-regressions
# into interpretable quantities:
#
#   1. Adjusted coefficient at GDP = -1 SD
#   2. Adjusted coefficient at GDP = mean
#   3. Adjusted coefficient at GDP = +1 SD
#
# marginalized over the observed region distribution.
#
# Also calculate:
#
#   4. Region-specific adjusted coefficients at mean GDP
#
# These are descriptive/model-based estimates from the
# already-frozen joint model.
#
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import t as student_t


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


INPUT_PATH = (
    OUT_DIR
    / "stage2_meta_analysis_input.csv"
)


PRIMARY_PATH = (
    OUT_DIR
    / "stage2_joint_gdp_region_primary30.csv"
)


COEFFICIENT_PATH = (
    OUT_DIR
    / "stage2_joint_gdp_region_coefficients.csv"
)


GINI_PATH = (
    OUT_DIR
    / "stage2_gini_incremental_after_gdp_region.csv"
)


ROBUSTNESS_PATH = (
    OUT_DIR
    / "stage2_gdp_region_north_america_robustness.csv"
)


# ============================================================
# 2. Load
# ============================================================

stage2 = pd.read_csv(
    INPUT_PATH
)


primary = pd.read_csv(
    PRIMARY_PATH
)


coefficients = pd.read_csv(
    COEFFICIENT_PATH
)


gini = pd.read_csv(
    GINI_PATH
)


robustness = pd.read_csv(
    ROBUSTNESS_PATH
)


# ============================================================
# 3. Frozen variables
# ============================================================

OUTCOMES = [

    "BEINGBULLIED",

    "BULLY_VERBAL",

    "BULLY_THREAT",

    "BULLY_PHYSICAL",

    "BULLY_RELATIONAL",
]


FOCAL_TERMS = [

    "BELONG",

    "DISCLIMA",

    "ST004D01T",
]


REGION_COL = (
    "WB_Region"
)


GDP_COL = (
    "LogGDPpcPPP_2018"
)


# ============================================================
# 4. Recover coefficient vectors for each joint model
#
# The joint model was:
#
# Stage1Coefficient
#     ~ Intercept
#     + LogGDP_z
#     + Region dummies
#
# GDP z=0 therefore corresponds to mean log GDP within the
# available systems.
# ============================================================

interpretation_rows = []

region_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp_coef = (

            coefficients[
                (
                    coefficients[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    coefficients[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .copy()
        )


        temp_data = (

            stage2[
                (
                    stage2[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    stage2[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
            .dropna(
                subset=[
                    GDP_COL,
                    REGION_COL,
                ]
            )
            .copy()
        )


        # ----------------------------------------------------
        # Extract coefficients
        # ----------------------------------------------------

        intercept_row = (

            temp_coef[
                temp_coef[
                    "MetaTerm"
                ]
                ==
                "Intercept"
            ]
        )


        gdp_row = (

            temp_coef[
                temp_coef[
                    "MetaTerm"
                ]
                ==
                "LogGDP_z"
            ]
        )


        if len(
            intercept_row
        ) != 1:

            raise RuntimeError(
                f"Missing intercept for "
                f"{outcome} {term}"
            )


        if len(
            gdp_row
        ) != 1:

            raise RuntimeError(
                f"Missing GDP slope for "
                f"{outcome} {term}"
            )


        intercept = float(
            intercept_row[
                "Estimate"
            ]
            .iloc[
                0
            ]
        )


        gdp_beta = float(
            gdp_row[
                "Estimate"
            ]
            .iloc[
                0
            ]
        )


        df = int(
            gdp_row[
                "DF"
            ]
            .iloc[
                0
            ]
        )


        reference_region = (

            intercept_row[
                "ReferenceRegion"
            ]
            .iloc[
                0
            ]
        )


        # ----------------------------------------------------
        # Region-specific adjusted coefficients at GDP z=0
        # ----------------------------------------------------

        region_counts = (

            temp_data[
                REGION_COL
            ]
            .value_counts()
        )


        region_estimates = {}


        for region, n_region in (
            region_counts.items()
        ):

            if region == reference_region:

                region_estimate = (
                    intercept
                )


            else:

                meta_name = (
                    f"Region::{region}"
                )


                row = (

                    temp_coef[
                        temp_coef[
                            "MetaTerm"
                        ]
                        ==
                        meta_name
                    ]
                )


                if len(
                    row
                ) != 1:

                    raise RuntimeError(
                        f"Missing region coefficient "
                        f"{meta_name} for "
                        f"{outcome} {term}"
                    )


                region_estimate = (

                    intercept

                    +

                    float(
                        row[
                            "Estimate"
                        ]
                        .iloc[
                            0
                        ]
                    )
                )


            region_estimates[
                region
            ] = (
                region_estimate
            )


            region_rows.append(
                {

                    "Outcome":
                        outcome,

                    "Term":
                        term,

                    "Region":
                        region,

                    "N_Systems":
                        int(
                            n_region
                        ),

                    "AdjustedCoefficient_AtMeanGDP":
                        region_estimate,

                    "ReferenceRegion":
                        reference_region,
                }
            )


        # ----------------------------------------------------
        # Region-marginalized intercept
        #
        # Weight each region equally to its observed number
        # of systems in this available-data model.
        # ----------------------------------------------------

        total_systems = float(
            region_counts.sum()
        )


        marginal_at_mean_gdp = 0.0


        for region, n_region in (
            region_counts.items()
        ):

            marginal_at_mean_gdp += (

                (
                    n_region
                    /
                    total_systems
                )

                *

                region_estimates[
                    region
                ]
            )


        # ----------------------------------------------------
        # GDP -1 SD, mean, +1 SD
        #
        # Since no GDP × region interaction is present,
        # GDP changes the coefficient by the same amount
        # within every region.
        # ----------------------------------------------------

        for gdp_z, gdp_label in [

            (
                -1.0,
                "LowGDP_minus1SD"
            ),

            (
                0.0,
                "MeanGDP"
            ),

            (
                1.0,
                "HighGDP_plus1SD"
            ),
        ]:

            predicted_beta = (

                marginal_at_mean_gdp

                +

                gdp_beta

                *
                gdp_z
            )


            if outcome == "BEINGBULLIED":

                predicted_or = (
                    np.nan
                )


            else:

                predicted_or = float(
                    np.exp(
                        predicted_beta
                    )
                )


            interpretation_rows.append(
                {

                    "Outcome":
                        outcome,

                    "Term":
                        term,

                    "GDP_Level":
                        gdp_label,

                    "GDP_Z":
                        gdp_z,

                    "PredictedCoefficient":
                        predicted_beta,

                    "PredictedOR":
                        predicted_or,

                    "GDP_ModerationBeta":
                        gdp_beta,

                    "N_Systems":
                        int(
                            len(
                                temp_data
                            )
                        ),

                    "ReferenceRegion":
                        reference_region,
                }
            )


# ============================================================
# 5. Tables
# ============================================================

gdp_interpretation = pd.DataFrame(
    interpretation_rows
)


region_interpretation = pd.DataFrame(
    region_rows
)


# ============================================================
# 6. Calculate high-vs-low GDP contrast
#
# +1 SD minus -1 SD = 2 × GDP moderator coefficient
# ============================================================

gdp_contrast_rows = []


for outcome in OUTCOMES:

    for term in FOCAL_TERMS:

        temp = (

            gdp_interpretation[
                (
                    gdp_interpretation[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    gdp_interpretation[
                        "Term"
                    ]
                    ==
                    term
                )
            ]
        )


        low = float(

            temp.loc[
                temp[
                    "GDP_Z"
                ]
                ==
                -1,
                "PredictedCoefficient"
            ]
            .iloc[
                0
            ]
        )


        high = float(

            temp.loc[
                temp[
                    "GDP_Z"
                ]
                ==
                1,
                "PredictedCoefficient"
            ]
            .iloc[
                0
            ]
        )


        gdp_test = (

            primary[
                (
                    primary[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
                &
                (
                    primary[
                        "Term"
                    ]
                    ==
                    term
                )
                &
                (
                    primary[
                        "Test"
                    ]
                    ==
                    "LogGDP_adjusted_for_Region"
                )
            ]
        )


        if len(
            gdp_test
        ) != 1:

            raise RuntimeError(
                "GDP primary test not found."
            )


        gdp_contrast_rows.append(
            {

                "Outcome":
                    outcome,

                "Term":
                    term,

                "LowGDP_Coefficient":
                    low,

                "HighGDP_Coefficient":
                    high,

                "HighMinusLow":
                    (
                        high
                        -
                        low
                    ),

                "GDP_Beta_Per1SD":
                    float(
                        gdp_test[
                            "Estimate"
                        ]
                        .iloc[
                            0
                        ]
                    ),

                "P":
                    float(
                        gdp_test[
                            "P"
                        ]
                        .iloc[
                            0
                        ]
                    ),

                "Q_FDR30":
                    float(
                        gdp_test[
                            "Q_FDR30"
                        ]
                        .iloc[
                            0
                        ]
                    ),

                "PrimaryFDR05":
                    bool(
                        gdp_test[
                            "FDR05"
                        ]
                        .iloc[
                            0
                        ]
                    ),
            }
        )


gdp_contrast = pd.DataFrame(
    gdp_contrast_rows
)


# ============================================================
# 7. Robust moderation classification
#
# Strong:
#   significant in primary model AND same FDR conclusion
#   after excluding North America.
#
# Borderline:
#   primary significant but loses significance only in the
#   North-America-excluded sensitivity.
# ============================================================

gdp_region_freeze = (

    primary
    .merge(

        robustness[
            [
                "Outcome",
                "Term",
                "Test",
                "PrimaryFDR05",
                "NoNorthAmericaFDR05",
                "SameFDRConclusion",
            ]
        ],

        on=[
            "Outcome",
            "Term",
            "Test",
        ],

        how="left",

        validate="one_to_one",
    )
)


gdp_region_freeze[
    "FinalClassification"
] = np.select(

    [

        (
            gdp_region_freeze[
                "FDR05"
            ]
            &
            gdp_region_freeze[
                "NoNorthAmericaFDR05"
            ]
        ),

        (
            gdp_region_freeze[
                "FDR05"
            ]
            &
            ~gdp_region_freeze[
                "NoNorthAmericaFDR05"
            ]
        ),

    ],

    [

        "Robust_FDR",

        "Borderline_Sensitivity",

    ],

    default=
        "Not_FDR_Significant",
)


# ============================================================
# 8. Gini freeze classification
# ============================================================

gini_freeze = (

    gini[
        [
            "Outcome",
            "Term",
            "N_Systems",
            "GiniBeta",
            "P",
            "Q_FDR15",
            "IncrementalPseudoR2",
            "FDR05",
        ]
    ]
    .copy()
)


gini_freeze[
    "FinalClassification"
] = np.where(

    gini_freeze[
        "FDR05"
    ],

    "FDR_Significant",

    "No_Independent_FDR_Evidence",
)


# ============================================================
# 9. Save final moderation interpretation tables
# ============================================================

gdp_interpretation.to_csv(
    OUT_DIR
    / "stage2_gdp_adjusted_predicted_coefficients.csv",
    index=False
)


gdp_contrast.to_csv(
    OUT_DIR
    / "stage2_gdp_high_low_contrasts.csv",
    index=False
)


region_interpretation.to_csv(
    OUT_DIR
    / "stage2_adjusted_region_coefficients_at_mean_gdp.csv",
    index=False
)


gdp_region_freeze.to_csv(
    OUT_DIR
    / "stage2_primary_moderation_frozen_classification.csv",
    index=False
)


gini_freeze.to_csv(
    OUT_DIR
    / "stage2_gini_frozen_classification.csv",
    index=False
)


# ============================================================
# 10. Print GDP interpretation
# ============================================================

print("\n")
print("=" * 130)
print("GDP-ADJUSTED MODERATION — INTERPRETABLE COEFFICIENTS")
print("=" * 130)


print(
    gdp_contrast
    .sort_values(
        [
            "Q_FDR30",
            "Outcome",
            "Term",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Robust moderation findings
# ============================================================

print("\n")
print("=" * 130)
print("FINAL GDP/REGION MODERATION CLASSIFICATION")
print("=" * 130)


print(
    gdp_region_freeze[
        [
            "Outcome",
            "Term",
            "Test",
            "Estimate",
            "P",
            "Q_FDR30",
            "NoNorthAmericaFDR05",
            "FinalClassification",
        ]
    ]
    .sort_values(
        [
            "FinalClassification",
            "Q_FDR30",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 12. Borderline sensitivity findings
# ============================================================

print("\n")
print("=" * 130)
print("BORDERLINE FINDINGS")
print("=" * 130)


borderline = (

    gdp_region_freeze[
        gdp_region_freeze[
            "FinalClassification"
        ]
        ==
        "Borderline_Sensitivity"
    ]
)


if len(
    borderline
) == 0:

    print(
        "None."
    )


else:

    print(
        borderline[
            [
                "Outcome",
                "Term",
                "Test",
                "Estimate",
                "P",
                "Q_FDR30",
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 13. Region estimates at mean GDP
# ============================================================

print("\n")
print("=" * 130)
print("ADJUSTED REGION-SPECIFIC COEFFICIENTS AT MEAN GDP")
print("=" * 130)


print(
    region_interpretation
    .sort_values(
        [
            "Outcome",
            "Term",
            "AdjustedCoefficient_AtMeanGDP",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 14. Gini conclusion
# ============================================================

print("\n")
print("=" * 130)
print("FINAL GINI MODERATION CLASSIFICATION")
print("=" * 130)


print(
    gini_freeze
    .sort_values(
        [
            "Q_FDR15",
            "Outcome",
            "Term",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 15. Freeze summary
# ============================================================

print("\n")
print("=" * 130)
print("PHASE 4 MODERATION FREEZE SUMMARY")
print("=" * 130)


print(
    "Primary GDP/region tests:",
    len(
        gdp_region_freeze
    )
)


print(
    "Robust FDR-significant:",
    int(
        (
            gdp_region_freeze[
                "FinalClassification"
            ]
            ==
            "Robust_FDR"
        )
        .sum()
    )
)


print(
    "Borderline sensitivity:",
    int(
        (
            gdp_region_freeze[
                "FinalClassification"
            ]
            ==
            "Borderline_Sensitivity"
        )
        .sum()
    )
)


print(
    "Not FDR-significant:",
    int(
        (
            gdp_region_freeze[
                "FinalClassification"
            ]
            ==
            "Not_FDR_Significant"
        )
        .sum()
    )
)


print(
    "\nIndependent Gini FDR-significant:",
    int(
        gini_freeze[
            "FDR05"
        ]
        .sum()
    ),
    "/",
    len(
        gini_freeze
    )
)


print(
    "\nNo new hypothesis testing was introduced in Phase 4E."
)


print(
    "Moderation analysis can now be frozen."
)



GDP-ADJUSTED MODERATION — INTERPRETABLE COEFFICIENTS
         Outcome      Term  LowGDP_Coefficient  HighGDP_Coefficient  HighMinusLow  GDP_Beta_Per1SD       P  Q_FDR30  PrimaryFDR05
  BULLY_PHYSICAL ST004D01T             0.71267              0.92941       0.21675          0.10837 0.00001  0.00008          True
BULLY_RELATIONAL    BELONG            -0.51323             -0.68563      -0.17240         -0.08620 0.00002  0.00010          True
    BULLY_THREAT  DISCLIMA            -0.25626             -0.35488      -0.09862         -0.04931 0.00002  0.00010          True
    BULLY_THREAT    BELONG            -0.52976             -0.72420      -0.19444         -0.09722 0.00003  0.00013          True
    BEINGBULLIED    BELONG            -0.24037             -0.29450      -0.05414         -0.02707 0.00004  0.00018          True
  BULLY_PHYSICAL  DISCLIMA            -0.29730             -0.34775      -0.05045         -0.02522 0.00372  0.00928          True
    BULLY_VERBAL  DISCLIMA         

In [1]:
# ============================================================
# PHASE 4F — MEDIATION / INDIRECT-ASSOCIATION FEASIBILITY AUDIT
# ============================================================
#
# EXPLORATORY PATHWAY
# -------------------
#
#   DISCLIMA  ->  BELONG  ->  BEINGBULLIED
#
#
# IMPORTANT
# ---------
# This is NOT yet a mediation analysis.
#
# We first audit whether:
#
#   1. the same complete-case sample is viable internationally
#   2. DISCLIMA is consistently associated with BELONG
#   3. BELONG remains associated with BEINGBULLIED after
#      adjustment for DISCLIMA
#   4. the direct DISCLIMA coefficient remains estimable
#
#
# Covariates:
#
#   AGE
#   ESCS
#   ST004D01T
#
#
# All continuous variables are standardized within system
# using survey-weighted means / SDs.
#
# Survey weights are used.
#
# This stage is descriptive/feasibility only.
# No indirect-effect significance test yet.
# ============================================================


from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd

import statsmodels.api as sm


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. Variables
# ============================================================

OUTCOME = "BEINGBULLIED"

EXPOSURE = "DISCLIMA"

MEDIATOR = "BELONG"


COVARIATES = [

    "AGE",

    "ESCS",

    "ST004D01T",
]


CONTINUOUS = [

    EXPOSURE,

    MEDIATOR,

    "AGE",

    "ESCS",
]


SCHOOL_COL = "CNTSCHID"

WEIGHT_COL = "W_FSTUWT"


# ============================================================
# 3. Recover frozen system universe
# ============================================================

rankings = pd.read_csv(

    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


SYSTEMS = sorted(

    rankings[
        "Country"
    ]
    .dropna()
    .astype(
        str
    )
    .unique()
)


assert len(
    SYSTEMS
) == 75


# ============================================================
# 4. Helpers
# ============================================================

def numeric_series(
    series
):

    output = pd.to_numeric(
        series,
        errors="coerce"
    ).astype(
        float
    )


    output[
        ~np.isfinite(
            output
        )
    ] = np.nan


    return output


def weighted_mean_sd(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    valid = (

        np.isfinite(
            values
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )
    )


    values = values[
        valid
    ]


    weights = weights[
        valid
    ]


    mean = float(
        np.average(
            values,
            weights=weights
        )
    )


    variance = float(

        np.average(

            (
                values
                -
                mean
            )
            **
            2,

            weights=weights
        )
    )


    return (
        mean,
        float(
            np.sqrt(
                variance
            )
        ),
    )


# ============================================================
# 5. Containers
# ============================================================

audit_rows = []

path_rows = []


# ============================================================
# 6. System loop
# ============================================================

for index, country in enumerate(
    SYSTEMS,
    start=1
):

    path = (

        PROCESSED_DIR

        /

        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------

    variables = (

        [
            OUTCOME,
            EXPOSURE,
            MEDIATOR,
            WEIGHT_COL,
        ]

        +

        COVARIATES
    )


    numeric = pd.DataFrame(
        index=df.index
    )


    for variable in variables:

        numeric[
            variable
        ] = numeric_series(
            df[
                variable
            ]
        )


    numeric[
        SCHOOL_COL
    ] = df[
        SCHOOL_COL
    ]


    # --------------------------------------------------------
    # Complete mediation-path sample
    # --------------------------------------------------------

    complete = pd.Series(
        True,
        index=df.index
    )


    for variable in variables:

        complete &= (
            numeric[
                variable
            ]
            .notna()
        )


    complete &= (

        numeric[
            WEIGHT_COL
        ]

        >
        0
    )


    complete &= (

        numeric[
            SCHOOL_COL
        ]
        .notna()
    )


    data = (

        numeric.loc[
            complete
        ]
        .copy()
    )


    # Expected:
    # Korea has no usable BEINGBULLIED
    if len(
        data
    ) == 0:

        if country == "KOR":

            print(
                f"[{index:02d}/75] "
                f"{country} — no BEINGBULLIED"
            )


            continue


        raise RuntimeError(
            f"No usable mediation sample for {country}"
        )


    # --------------------------------------------------------
    # Normalize weights
    # --------------------------------------------------------

    weights = (

        data[
            WEIGHT_COL
        ]
        .to_numpy(
            dtype=float
        )
    )


    weights = (

        weights

        /

        np.mean(
            weights
        )
    )


    # --------------------------------------------------------
    # Weighted within-system standardization
    # --------------------------------------------------------

    standardized = pd.DataFrame(
        index=data.index
    )


    for variable in CONTINUOUS:

        mean, sd = weighted_mean_sd(

            data[
                variable
            ],

            weights
        )


        if (
            not np.isfinite(
                sd
            )
            or
            sd
            <=
            0
        ):

            raise RuntimeError(
                f"Invalid SD for {country} {variable}"
            )


        standardized[
            variable
        ] = (

            data[
                variable
            ]

            -

            mean

        ) / sd


    standardized[
        "ST004D01T"
    ] = data[
        "ST004D01T"
    ].astype(
        float
    )


    standardized[
        OUTCOME
    ] = data[
        OUTCOME
    ].astype(
        float
    )


    # ========================================================
    # A PATH
    #
    # BELONG ~ DISCLIMA + AGE + ESCS + gender
    # ========================================================

    X_a = standardized[
        [
            EXPOSURE,
            "AGE",
            "ESCS",
            "ST004D01T",
        ]
    ]


    X_a = sm.add_constant(
        X_a,
        has_constant="add"
    )


    model_a = sm.WLS(

        standardized[
            MEDIATOR
        ],

        X_a,

        weights=weights,
    )


    fit_a = model_a.fit()


    a_beta = float(
        fit_a.params[
            EXPOSURE
        ]
    )


    # ========================================================
    # B + C' PATH
    #
    # BEINGBULLIED ~
    # BELONG + DISCLIMA + AGE + ESCS + gender
    # ========================================================

    X_b = standardized[
        [
            MEDIATOR,
            EXPOSURE,
            "AGE",
            "ESCS",
            "ST004D01T",
        ]
    ]


    X_b = sm.add_constant(
        X_b,
        has_constant="add"
    )


    model_b = sm.WLS(

        standardized[
            OUTCOME
        ],

        X_b,

        weights=weights,
    )


    fit_b = model_b.fit()


    b_beta = float(
        fit_b.params[
            MEDIATOR
        ]
    )


    c_prime = float(
        fit_b.params[
            EXPOSURE
        ]
    )


    # ========================================================
    # TOTAL C PATH
    #
    # BEINGBULLIED ~
    # DISCLIMA + AGE + ESCS + gender
    # ========================================================

    X_c = standardized[
        [
            EXPOSURE,
            "AGE",
            "ESCS",
            "ST004D01T",
        ]
    ]


    X_c = sm.add_constant(
        X_c,
        has_constant="add"
    )


    model_c = sm.WLS(

        standardized[
            OUTCOME
        ],

        X_c,

        weights=weights,
    )


    fit_c = model_c.fit()


    c_total = float(
        fit_c.params[
            EXPOSURE
        ]
    )


    # ========================================================
    # Descriptive product only
    #
    # NO inference yet.
    # ========================================================

    indirect_product = (
        a_beta
        *
        b_beta
    )


    path_rows.append(
        {

            "Country":
                country,

            "N":
                len(
                    data
                ),

            "N_Schools":
                int(
                    data[
                        SCHOOL_COL
                    ]
                    .nunique()
                ),

            "A_DISCLIMA_to_BELONG":
                a_beta,

            "B_BELONG_to_BULLYING":
                b_beta,

            "Cprime_DISCLIMA_to_BULLYING":
                c_prime,

            "Ctotal_DISCLIMA_to_BULLYING":
                c_total,

            "IndirectProduct_AxB":
                indirect_product,
        }
    )


    audit_rows.append(
        {

            "Country":
                country,

            "N":
                len(
                    data
                ),

            "N_Schools":
                int(
                    data[
                        SCHOOL_COL
                    ]
                    .nunique()
                ),

            "A_Positive":
                (
                    a_beta
                    >
                    0
                ),

            "B_Negative":
                (
                    b_beta
                    <
                    0
                ),

            "IndirectNegative":
                (
                    indirect_product
                    <
                    0
                ),

            "DirectNegative":
                (
                    c_prime
                    <
                    0
                ),

            "TotalNegative":
                (
                    c_total
                    <
                    0
                ),
        }
    )


    print(
        f"[{index:02d}/75] "
        f"{country}"
        f" | N={len(data)}"
        f" | a={a_beta:.4f}"
        f" | b={b_beta:.4f}"
        f" | a×b={indirect_product:.4f}"
    )


    del (
        df,
        numeric,
        data,
        standardized
    )


    gc.collect()


# ============================================================
# 7. DataFrames
# ============================================================

paths = pd.DataFrame(
    path_rows
)


audit = pd.DataFrame(
    audit_rows
)


assert len(
    paths
) == 74


# ============================================================
# 8. Save
# ============================================================

paths.to_csv(

    OUT_DIR
    / "mediation_feasibility_system_paths.csv",

    index=False
)


audit.to_csv(

    OUT_DIR
    / "mediation_feasibility_sign_audit.csv",

    index=False
)


# ============================================================
# 9. Summary
# ============================================================

summary = pd.DataFrame(
    {

        "Metric": [

            "Systems",

            "Median N",

            "Minimum N",

            "Median schools",

            "A path positive",

            "B path negative",

            "Indirect product negative",

            "Direct path negative",

            "Total path negative",
        ],

        "Value": [

            len(
                paths
            ),

            paths[
                "N"
            ]
            .median(),

            paths[
                "N"
            ]
            .min(),

            paths[
                "N_Schools"
            ]
            .median(),

            (
                paths[
                    "A_DISCLIMA_to_BELONG"
                ]
                >
                0
            )
            .sum(),

            (
                paths[
                    "B_BELONG_to_BULLYING"
                ]
                <
                0
            )
            .sum(),

            (
                paths[
                    "IndirectProduct_AxB"
                ]
                <
                0
            )
            .sum(),

            (
                paths[
                    "Cprime_DISCLIMA_to_BULLYING"
                ]
                <
                0
            )
            .sum(),

            (
                paths[
                    "Ctotal_DISCLIMA_to_BULLYING"
                ]
                <
                0
            )
            .sum(),
        ],
    }
)


summary.to_csv(

    OUT_DIR
    / "mediation_feasibility_summary.csv",

    index=False
)


# ============================================================
# 10. Distribution summary
# ============================================================

distribution = (

    paths[
        [
            "A_DISCLIMA_to_BELONG",
            "B_BELONG_to_BULLYING",
            "Cprime_DISCLIMA_to_BULLYING",
            "Ctotal_DISCLIMA_to_BULLYING",
            "IndirectProduct_AxB",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max",
        ]
    )
    .T
)


# ============================================================
# 11. Print
# ============================================================

print("\n")
print("=" * 125)
print("MEDIATION FEASIBILITY SUMMARY")
print("=" * 125)


print(
    summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 125)
print("SYSTEM-LEVEL PATH DISTRIBUTIONS")
print("=" * 125)


print(
    distribution
    .round(
        5
    )
    .to_string()
)


print("\n")
print("=" * 125)
print("SYSTEMS WITH NON-DOMINANT PATH DIRECTIONS")
print("=" * 125)


exceptions = (

    paths[
        (
            paths[
                "A_DISCLIMA_to_BELONG"
            ]
            <=
            0
        )
        |
        (
            paths[
                "B_BELONG_to_BULLYING"
            ]
            >=
            0
        )
        |
        (
            paths[
                "IndirectProduct_AxB"
            ]
            >=
            0
        )
    ]
)


if len(
    exceptions
) == 0:

    print(
        "None."
    )


else:

    print(
        exceptions
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 125)
print("PHASE 4F MEDIATION FEASIBILITY AUDIT COMPLETE")
print("=" * 125)


print(
    "\nNo mediation/indirect-effect significance tests "
    "have been performed."
)


print(
    "The product a×b is descriptive only at this stage."
)

[01/75] ALB | N=5814 | a=0.1745 | b=-0.2026 | a×b=-0.0353
[02/75] ARE | N=16597 | a=0.2086 | b=-0.3188 | a×b=-0.0665
[03/75] ARG | N=7420 | a=0.1178 | b=-0.3442 | a×b=-0.0405
[04/75] AUS | N=10439 | a=0.1388 | b=-0.3835 | a×b=-0.0533
[05/75] AUT | N=5153 | a=0.1292 | b=-0.2640 | a×b=-0.0341
[06/75] BEL | N=7047 | a=0.0767 | b=-0.2849 | a×b=-0.0218
[07/75] BGR | N=3517 | a=0.1465 | b=-0.2949 | a×b=-0.0432
[08/75] BIH | N=5167 | a=0.1042 | b=-0.2244 | a×b=-0.0234
[09/75] BLR | N=5250 | a=0.1834 | b=-0.2091 | a×b=-0.0383
[10/75] BRA | N=6169 | a=0.1237 | b=-0.2734 | a×b=-0.0338
[11/75] BRN | N=4647 | a=0.1212 | b=-0.2178 | a×b=-0.0264
[12/75] CAN | N=18580 | a=0.0761 | b=-0.3171 | a×b=-0.0241
[13/75] CHE | N=3461 | a=0.0664 | b=-0.3057 | a×b=-0.0203
[14/75] CHL | N=4911 | a=0.1283 | b=-0.2245 | a×b=-0.0288
[15/75] COL | N=5080 | a=0.1471 | b=-0.2129 | a×b=-0.0313
[16/75] CRI | N=6115 | a=0.1150 | b=-0.2992 | a×b=-0.0344
[17/75] CZE | N=6023 | a=0.1184 | b=-0.2790 | a×b=-0.0330
[18/75] DEU

In [2]:
# ============================================================
# PHASE 4G0 — PISA REPLICATE-WEIGHT AUDIT FOR
# INDIRECT-ASSOCIATION INFERENCE
# ============================================================
#
# PURPOSE
# -------
# Determine whether the processed PISA files retain all
# 80 official student replicate weights:
#
#       W_FSTR1 ... W_FSTR80
#
# If they do, we will use PISA's replicate-weight design for
# the indirect-association uncertainty rather than an ad hoc
# school-cluster bootstrap.
#
# NO MODELS ARE FITTED HERE.
# ============================================================


from pathlib import Path
import re

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FEASIBILITY_PATH = (
    OUT_DIR
    / "mediation_feasibility_system_paths.csv"
)


# ============================================================
# 2. Systems with usable BEINGBULLIED
# ============================================================

feasibility = pd.read_csv(
    FEASIBILITY_PATH
)


SYSTEMS = sorted(
    feasibility[
        "Country"
    ]
    .unique()
)


assert len(
    SYSTEMS
) == 74


# ============================================================
# 3. Expected replicate weights
# ============================================================

EXPECTED_REPLICATE_WEIGHTS = [

    f"W_FSTR{i}"

    for i in range(
        1,
        81
    )
]


FULL_WEIGHT = (
    "W_FSTUWT"
)


# ============================================================
# 4. Mediation-path variables
# ============================================================

REQUIRED_PATH_COLUMNS = [

    "BEINGBULLIED",

    "DISCLIMA",

    "BELONG",

    "AGE",

    "ESCS",

    "ST004D01T",

    "CNTSCHID",

    FULL_WEIGHT,
]


# ============================================================
# 5. Audit
# ============================================================

audit_rows = []


for index, country in enumerate(
    SYSTEMS,
    start=1
):

    path = (

        PROCESSED_DIR

        /

        f"{country}_analysis.pkl"
    )


    df = pd.read_pickle(
        path
    )


    columns = set(
        df.columns
    )


    # --------------------------------------------------------
    # Detect replicate weights actually present
    # --------------------------------------------------------

    detected = sorted(
        [

            column

            for column
            in df.columns

            if re.fullmatch(
                r"W_FSTR\d+",
                str(
                    column
                )
            )
        ],

        key=lambda x:
            int(
                re.search(
                    r"\d+",
                    str(
                        x
                    )
                )
                .group()
            )
    )


    missing_expected = [

        column

        for column
        in EXPECTED_REPLICATE_WEIGHTS

        if column
        not in columns
    ]


    # --------------------------------------------------------
    # Same complete-case sample as Phase 4F
    # --------------------------------------------------------

    complete = pd.Series(
        True,
        index=df.index
    )


    for variable in [

        "BEINGBULLIED",

        "DISCLIMA",

        "BELONG",

        "AGE",

        "ESCS",

        "ST004D01T",

        FULL_WEIGHT,
    ]:

        values = pd.to_numeric(
            df[
                variable
            ],
            errors="coerce"
        )


        complete &= (

            values.notna()

            &

            np.isfinite(
                values
            )
        )


    complete &= (

        pd.to_numeric(
            df[
                FULL_WEIGHT
            ],
            errors="coerce"
        )

        >
        0
    )


    complete &= (

        df[
            "CNTSCHID"
        ]
        .notna()
    )


    n_complete = int(
        complete.sum()
    )


    # --------------------------------------------------------
    # Replicate-weight quality on actual analytic sample
    # --------------------------------------------------------

    replicate_stats = []


    for replicate_weight in detected:

        values = pd.to_numeric(

            df.loc[
                complete,
                replicate_weight
            ],

            errors="coerce",
        )


        valid = (

            values.notna()

            &

            np.isfinite(
                values
            )

            &

            (
                values
                >=
                0
            )
        )


        replicate_stats.append(
            {

                "Column":
                    replicate_weight,

                "NValid":
                    int(
                        valid.sum()
                    ),

                "NPositive":
                    int(
                        (
                            values
                            >
                            0
                        )
                        .sum()
                    ),

                "SumWeight":
                    float(
                        values[
                            valid
                        ]
                        .sum()
                    )
                    if valid.any()
                    else np.nan,

                "NUnique":
                    int(
                        values[
                            valid
                        ]
                        .nunique()
                    )
                    if valid.any()
                    else 0,
            }
        )


    if len(
        replicate_stats
    ) > 0:

        rep_stats = pd.DataFrame(
            replicate_stats
        )


        min_valid = int(
            rep_stats[
                "NValid"
            ]
            .min()
        )


        min_positive = int(
            rep_stats[
                "NPositive"
            ]
            .min()
        )


        min_unique = int(
            rep_stats[
                "NUnique"
            ]
            .min()
        )


        max_unique = int(
            rep_stats[
                "NUnique"
            ]
            .max()
        )


    else:

        min_valid = 0

        min_positive = 0

        min_unique = 0

        max_unique = 0


    audit_rows.append(
        {

            "Country":
                country,

            "N_CompleteCase":
                n_complete,

            "N_ReplicateWeightsDetected":
                len(
                    detected
                ),

            "All80Present":
                (
                    len(
                        missing_expected
                    )
                    ==
                    0
                ),

            "MissingReplicateWeights":
                "|".join(
                    missing_expected
                ),

            "MinValidAcrossReplicates":
                min_valid,

            "MinPositiveAcrossReplicates":
                min_positive,

            "MinUniqueWeightValues":
                min_unique,

            "MaxUniqueWeightValues":
                max_unique,
        }
    )


    print(
        f"[{index:02d}/74] "
        f"{country}"
        f" | replicate weights={len(detected)}"
        f" | all80={len(missing_expected) == 0}"
        f" | N={n_complete}"
    )


# ============================================================
# 6. Results
# ============================================================

audit = pd.DataFrame(
    audit_rows
)


audit.to_csv(
    OUT_DIR
    / "mediation_replicate_weight_audit.csv",
    index=False
)


# ============================================================
# 7. Summary
# ============================================================

print("\n")
print("=" * 125)
print("PISA REPLICATE-WEIGHT AUDIT")
print("=" * 125)


print(
    "Systems audited:",
    len(
        audit
    )
)


print(
    "Systems with all 80 replicate weights:",
    int(
        audit[
            "All80Present"
        ]
        .sum()
    ),
    "/",
    len(
        audit
    )
)


print(
    "\nReplicate-weight counts:"
)


print(
    audit[
        "N_ReplicateWeightsDetected"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\nMinimum valid replicate-weight observations "
    "across all systems:"
)


print(
    audit[
        "MinValidAcrossReplicates"
    ]
    .min()
)


print(
    "\nMinimum positive replicate-weight observations:"
)


print(
    audit[
        "MinPositiveAcrossReplicates"
    ]
    .min()
)


# ============================================================
# 8. Problems
# ============================================================

problems = (

    audit[
        (
            ~audit[
                "All80Present"
            ]
        )
        |
        (
            audit[
                "MinValidAcrossReplicates"
            ]
            <
            audit[
                "N_CompleteCase"
            ]
        )
        |
        (
            audit[
                "MinUniqueWeightValues"
            ]
            <=
            1
        )
    ]
)


print("\n")
print("=" * 125)
print("REPLICATE-WEIGHT PROBLEMS")
print("=" * 125)


if len(
    problems
) == 0:

    print(
        "None."
    )


else:

    print(
        problems
        .to_string(
            index=False
        )
    )


# ============================================================
# 9. Final decision
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 4G0 DECISION")
print("=" * 125)


if (
    len(
        audit
    )
    ==
    74
    and
    audit[
        "All80Present"
    ]
    .all()
    and
    (
        audit[
            "MinValidAcrossReplicates"
        ]
        >=
        audit[
            "N_CompleteCase"
        ]
    )
    .all()
):

    print(
        "All 74 systems retain the full PISA "
        "80-replicate-weight design."
    )


    print(
        "Proceed with replicate-weight indirect-association "
        "variance estimation."
    )


else:

    print(
        "Full replicate-weight design is not available "
        "for every system."
    )


    print(
        "Use the preplanned school-cluster bootstrap fallback."
    )

[01/74] ALB | replicate weights=0 | all80=False | N=5814
[02/74] ARE | replicate weights=0 | all80=False | N=16597
[03/74] ARG | replicate weights=0 | all80=False | N=7420
[04/74] AUS | replicate weights=0 | all80=False | N=10439
[05/74] AUT | replicate weights=0 | all80=False | N=5153
[06/74] BEL | replicate weights=0 | all80=False | N=7047
[07/74] BGR | replicate weights=0 | all80=False | N=3517
[08/74] BIH | replicate weights=0 | all80=False | N=5167
[09/74] BLR | replicate weights=0 | all80=False | N=5250
[10/74] BRA | replicate weights=0 | all80=False | N=6169
[11/74] BRN | replicate weights=0 | all80=False | N=4647
[12/74] CAN | replicate weights=0 | all80=False | N=18580
[13/74] CHE | replicate weights=0 | all80=False | N=3461
[14/74] CHL | replicate weights=0 | all80=False | N=4911
[15/74] COL | replicate weights=0 | all80=False | N=5080
[16/74] CRI | replicate weights=0 | all80=False | N=6115
[17/74] CZE | replicate weights=0 | all80=False | N=6023
[18/74] DEU | replicate weig

In [4]:
# ============================================================
# RESTORE PHASE 4 META-ANALYSIS HELPERS
# ============================================================

import numpy as np
from scipy.optimize import minimize_scalar


# ============================================================
# REML random-effects meta-analysis / meta-regression
# ============================================================

def fit_reml(
    y,
    variance,
    X
):

    y = np.asarray(
        y,
        dtype=float
    )

    variance = np.asarray(
        variance,
        dtype=float
    )

    X = np.asarray(
        X,
        dtype=float
    )


    valid = (
        np.isfinite(y)
        &
        np.isfinite(variance)
        &
        (variance > 0)
        &
        np.isfinite(X).all(axis=1)
    )


    y = y[valid]

    variance = variance[valid]

    X = X[valid]


    k = len(y)

    p = X.shape[1]


    if k <= p:

        raise RuntimeError(
            "Not enough observations for REML model."
        )


    # --------------------------------------------------------
    # REML objective
    # --------------------------------------------------------

    def reml_objective(
        tau2
    ):

        total_variance = (
            variance
            +
            tau2
        )


        weights = (
            1.0
            /
            total_variance
        )


        XtWX = (
            X.T
            @
            (
                weights[:, None]
                *
                X
            )
        )


        sign, logdet = np.linalg.slogdet(
            XtWX
        )


        if sign <= 0:

            return np.inf


        covariance = np.linalg.pinv(
            XtWX
        )


        beta = (
            covariance
            @
            (
                X.T
                @
                (
                    weights
                    *
                    y
                )
            )
        )


        residual = (
            y
            -
            X
            @
            beta
        )


        weighted_residual_ss = float(
            np.sum(
                weights
                *
                residual ** 2
            )
        )


        objective = 0.5 * (
            np.sum(
                np.log(
                    total_variance
                )
            )
            +
            logdet
            +
            weighted_residual_ss
        )


        return float(
            objective
        )


    # --------------------------------------------------------
    # Adaptive tau² search range
    # --------------------------------------------------------

    observed_variance = float(
        np.var(
            y,
            ddof=1
        )
    )


    upper = max(
        1.0,
        observed_variance * 10.0,
        float(
            np.median(
                variance
            )
        ) * 100.0,
    )


    optimization = minimize_scalar(
        reml_objective,
        bounds=(
            0.0,
            upper
        ),
        method="bounded",
        options={
            "xatol": 1e-10
        },
    )


    if not optimization.success:

        raise RuntimeError(
            "REML optimization failed."
        )


    tau2_candidate = float(
        optimization.x
    )


    # Compare against boundary tau² = 0
    if (
        reml_objective(0.0)
        <=
        optimization.fun
    ):

        tau2 = 0.0

    else:

        tau2 = tau2_candidate


    # --------------------------------------------------------
    # Final fit
    # --------------------------------------------------------

    total_variance = (
        variance
        +
        tau2
    )


    weights = (
        1.0
        /
        total_variance
    )


    XtWX = (
        X.T
        @
        (
            weights[:, None]
            *
            X
        )
    )


    covariance = np.linalg.pinv(
        XtWX
    )


    beta = (
        covariance
        @
        (
            X.T
            @
            (
                weights
                *
                y
            )
        )
    )


    residual = (
        y
        -
        X
        @
        beta
    )


    return {
        "beta": beta,
        "covariance": covariance,
        "tau2": tau2,
        "weights": weights,
        "residual": residual,
        "k": k,
        "p": p,
    }


# ============================================================
# Modified Knapp-Hartung adjustment
# ============================================================

def modified_knapp_hartung(
    fit
):

    weights = np.asarray(
        fit["weights"],
        dtype=float
    )

    residual = np.asarray(
        fit["residual"],
        dtype=float
    )


    k = int(
        fit["k"]
    )

    p = int(
        fit["p"]
    )


    df_residual = (
        k
        -
        p
    )


    if df_residual <= 0:

        raise RuntimeError(
            "Invalid residual degrees of freedom."
        )


    Q_residual = float(
        np.sum(
            weights
            *
            residual ** 2
        )
    )


    raw_scale = (
        Q_residual
        /
        df_residual
    )


    # Conservative modification:
    # never shrink the conventional covariance
    scale = max(
        1.0,
        raw_scale
    )


    adjusted_covariance = (
        fit["covariance"]
        *
        scale
    )


    return {
        "covariance": adjusted_covariance,
        "df": df_residual,
        "Q_residual": Q_residual,
        "KH_raw_scale": raw_scale,
        "KH_scale": scale,
    }


print("Phase 4 meta-analysis helpers restored successfully.")

Phase 4 meta-analysis helpers restored successfully.


In [5]:
# ============================================================
# PHASE 4G1 — SCHOOL-CLUSTER BOOTSTRAP OF INDIRECT ASSOCIATION
#              + RANDOM-EFFECTS META-ANALYSIS
# ============================================================
#
# EXPLORATORY / NONCAUSAL PATHWAY
# -------------------------------
#
#   DISCLIMA  ->  BELONG  ->  BEINGBULLIED
#
#
# Within each education system:
#
#   a:
#     BELONG ~ DISCLIMA + AGE + ESCS + gender
#
#   b and c':
#     BEINGBULLIED ~
#       BELONG + DISCLIMA + AGE + ESCS + gender
#
#   c:
#     BEINGBULLIED ~
#       DISCLIMA + AGE + ESCS + gender
#
#   indirect association:
#
#       a × b
#
#
# UNCERTAINTY
# -----------
#
# Official PISA replicate weights were not retained in the
# processed analysis files.
#
# Therefore we use the predeclared fallback:
#
#   - resample SCHOOLS with replacement
#   - preserve students within sampled schools
#   - retain W_FSTUWT
#   - refit the complete estimator in each bootstrap
#   - recompute within-system weighted standardization
#
#
# BOOTSTRAPS
# ----------
#
# 1,000 per education system.
#
# Checkpointed by system so the cell is resumable.
#
#
# FINAL STAGE
# -----------
#
# System-specific a×b estimates and bootstrap variances are
# combined using REML random-effects meta-analysis with
# modified Knapp-Hartung inference.
#
#
# IMPORTANT
# ---------
#
# This is a CROSS-SECTIONAL INDIRECT ASSOCIATION.
#
# It is NOT evidence that BELONG causally mediates an effect
# of DISCLIMA on bullying.
# ============================================================


from pathlib import Path
import gc
import zlib

import numpy as np
import pandas as pd

from scipy.stats import (
    t as student_t,
    chi2,
)


# ============================================================
# 1. Paths
# ============================================================

PROCESSED_DIR = Path(
    "../data/processed_final"
)


RESULT_ROOT = Path(
    "../results"
)


OUT_DIR = (
    RESULT_ROOT
    / "multilevel_analysis"
)


CHECKPOINT_DIR = (
    OUT_DIR
    / "mediation_cluster_bootstrap_checkpoints"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FEASIBILITY_PATH = (
    OUT_DIR
    / "mediation_feasibility_system_paths.csv"
)


# ============================================================
# 2. Verify previous meta-analysis helpers
# ============================================================

for function_name in [

    "fit_reml",

    "modified_knapp_hartung",

]:

    if function_name not in globals():

        raise RuntimeError(

            f"{function_name} is missing. "
            "Run the Phase 4C / 4D cells first."
        )


# ============================================================
# 3. Frozen analysis specification
# ============================================================

OUTCOME = "BEINGBULLIED"

EXPOSURE = "DISCLIMA"

MEDIATOR = "BELONG"


AGE = "AGE"

ESCS = "ESCS"

GENDER = "ST004D01T"


SCHOOL_COL = "CNTSCHID"

WEIGHT_COL = "W_FSTUWT"


N_BOOTSTRAPS = 1000

BASE_SEED = 42


# ============================================================
# 4. Frozen usable system universe
# ============================================================

feasibility = pd.read_csv(
    FEASIBILITY_PATH
)


SYSTEMS = sorted(

    feasibility[
        "Country"
    ]
    .unique()
)


assert len(
    SYSTEMS
) == 74


feasibility_ref = (

    feasibility
    .set_index(
        "Country"
    )
)


print(
    "Systems:",
    len(
        SYSTEMS
    )
)


print(
    "Bootstrap replicates per system:",
    N_BOOTSTRAPS
)


# ============================================================
# 5. Numeric helper
# ============================================================

def numeric_array(
    series
):

    values = pd.to_numeric(
        series,
        errors="coerce"
    ).to_numpy(
        dtype=float
    )


    values[
        ~np.isfinite(
            values
        )
    ] = np.nan


    return values


# ============================================================
# 6. Weighted mean / SD
# ============================================================

def weighted_mean_sd_np(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    keep = (

        np.isfinite(
            values
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )
    )


    values = values[
        keep
    ]


    weights = weights[
        keep
    ]


    weight_sum = np.sum(
        weights
    )


    if weight_sum <= 0:

        raise RuntimeError(
            "Non-positive bootstrap weight sum."
        )


    mean = float(

        np.sum(
            weights
            *
            values
        )

        /

        weight_sum
    )


    variance = float(

        np.sum(

            weights

            *

            (
                values
                -
                mean
            )
            **
            2
        )

        /

        weight_sum
    )


    sd = float(
        np.sqrt(
            variance
        )
    )


    if (
        not np.isfinite(
            sd
        )
        or
        sd
        <=
        0
    ):

        raise RuntimeError(
            "Invalid weighted SD."
        )


    return mean, sd


# ============================================================
# 7. Fast weighted least squares
# ============================================================

def weighted_beta(
    X,
    y,
    weights
):

    X = np.asarray(
        X,
        dtype=float
    )


    y = np.asarray(
        y,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    keep = (

        np.isfinite(
            y
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )

        &

        np.isfinite(
            X
        )
        .all(
            axis=1
        )
    )


    X = X[
        keep
    ]


    y = y[
        keep
    ]


    weights = weights[
        keep
    ]


    sqrt_w = np.sqrt(
        weights
    )


    Xw = (

        X

        *

        sqrt_w[
            :,
            None
        ]
    )


    yw = (

        y

        *

        sqrt_w
    )


    beta = np.linalg.lstsq(

        Xw,

        yw,

        rcond=None,

    )[
        0
    ]


    return beta


# ============================================================
# 8. Fit pathway for one arbitrary weight vector
# ============================================================

def compute_path_estimates(
    raw,
    bootstrap_weights
):

    w = np.asarray(
        bootstrap_weights,
        dtype=float
    )


    positive = (
        w
        >
        0
    )


    if positive.sum() == 0:

        raise RuntimeError(
            "No positive bootstrap weights."
        )


    # --------------------------------------------------------
    # Scaling all weights by a constant does not alter WLS.
    # Normalize for numerical stability.
    # --------------------------------------------------------

    w = (

        w

        /

        np.mean(
            w[
                positive
            ]
        )
    )


    # --------------------------------------------------------
    # Recompute weighted standardization inside EACH
    # bootstrap replicate.
    # --------------------------------------------------------

    standardized = {}


    for variable in [

        EXPOSURE,

        MEDIATOR,

        AGE,

        ESCS,

    ]:

        mean, sd = weighted_mean_sd_np(

            raw[
                variable
            ],

            w,
        )


        standardized[
            variable
        ] = (

            raw[
                variable
            ]

            -

            mean

        ) / sd


    y = raw[
        OUTCOME
    ]


    gender = raw[
        GENDER
    ]


    n = len(
        y
    )


    intercept = np.ones(
        n,
        dtype=float
    )


    # ========================================================
    # A PATH
    #
    # BELONG_z ~
    # DISCLIMA_z + AGE_z + ESCS_z + gender
    # ========================================================

    X_a = np.column_stack(
        [

            intercept,

            standardized[
                EXPOSURE
            ],

            standardized[
                AGE
            ],

            standardized[
                ESCS
            ],

            gender,
        ]
    )


    beta_a = weighted_beta(

        X=X_a,

        y=standardized[
            MEDIATOR
        ],

        weights=w,
    )


    a = float(
        beta_a[
            1
        ]
    )


    # ========================================================
    # B + C' PATH
    #
    # BEINGBULLIED ~
    # BELONG_z + DISCLIMA_z + AGE_z + ESCS_z + gender
    # ========================================================

    X_b = np.column_stack(
        [

            intercept,

            standardized[
                MEDIATOR
            ],

            standardized[
                EXPOSURE
            ],

            standardized[
                AGE
            ],

            standardized[
                ESCS
            ],

            gender,
        ]
    )


    beta_b = weighted_beta(

        X=X_b,

        y=y,

        weights=w,
    )


    b = float(
        beta_b[
            1
        ]
    )


    c_prime = float(
        beta_b[
            2
        ]
    )


    # ========================================================
    # TOTAL C PATH
    #
    # BEINGBULLIED ~
    # DISCLIMA_z + AGE_z + ESCS_z + gender
    # ========================================================

    X_c = np.column_stack(
        [

            intercept,

            standardized[
                EXPOSURE
            ],

            standardized[
                AGE
            ],

            standardized[
                ESCS
            ],

            gender,
        ]
    )


    beta_c = weighted_beta(

        X=X_c,

        y=y,

        weights=w,
    )


    c_total = float(
        beta_c[
            1
        ]
    )


    indirect = float(
        a
        *
        b
    )


    decomposition_error = float(

        c_total

        -

        (
            c_prime
            +
            indirect
        )
    )


    return {

        "a":
            a,

        "b":
            b,

        "c_prime":
            c_prime,

        "c_total":
            c_total,

        "indirect":
            indirect,

        "decomposition_error":
            decomposition_error,
    }


# ============================================================
# 9. Main bootstrap
# ============================================================

for system_index, country in enumerate(
    SYSTEMS,
    start=1
):

    checkpoint_path = (

        CHECKPOINT_DIR

        /

        f"{country}.npz"
    )


    if checkpoint_path.exists():

        print(

            f"[{system_index:02d}/74] "
            f"{country} — checkpoint exists"
        )


        continue


    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    df = pd.read_pickle(

        PROCESSED_DIR

        /

        f"{country}_analysis.pkl"
    )


    numeric = {}


    for variable in [

        OUTCOME,

        EXPOSURE,

        MEDIATOR,

        AGE,

        ESCS,

        GENDER,

        WEIGHT_COL,

    ]:

        numeric[
            variable
        ] = numeric_array(
            df[
                variable
            ]
        )


    complete = np.ones(
        len(
            df
        ),
        dtype=bool
    )


    for variable in [

        OUTCOME,

        EXPOSURE,

        MEDIATOR,

        AGE,

        ESCS,

        GENDER,

        WEIGHT_COL,

    ]:

        complete &= np.isfinite(
            numeric[
                variable
            ]
        )


    complete &= (

        numeric[
            WEIGHT_COL
        ]

        >
        0
    )


    complete &= (

        df[
            SCHOOL_COL
        ]
        .notna()
        .to_numpy()
    )


    raw = {

        variable:
            numeric[
                variable
            ][
                complete
            ]

        for variable
        in [

            OUTCOME,

            EXPOSURE,

            MEDIATOR,

            AGE,

            ESCS,

            GENDER,

        ]
    }


    base_weights = (

        numeric[
            WEIGHT_COL
        ][
            complete
        ]
    )


    schools = (

        df.loc[
            complete,
            SCHOOL_COL
        ]
        .astype(
            str
        )
        .to_numpy()
    )


    unique_schools, school_codes = (
        np.unique(
            schools,
            return_inverse=True,
        )
    )


    G = len(
        unique_schools
    )


    N = len(
        base_weights
    )


    # --------------------------------------------------------
    # Full-sample point estimate
    # --------------------------------------------------------

    point = compute_path_estimates(

        raw=raw,

        bootstrap_weights=base_weights,
    )


    # --------------------------------------------------------
    # Verify exact agreement with Phase 4F
    # --------------------------------------------------------

    previous = feasibility_ref.loc[
        country
    ]


    point_differences = [

        abs(
            point[
                "a"
            ]
            -
            previous[
                "A_DISCLIMA_to_BELONG"
            ]
        ),

        abs(
            point[
                "b"
            ]
            -
            previous[
                "B_BELONG_to_BULLYING"
            ]
        ),

        abs(
            point[
                "c_prime"
            ]
            -
            previous[
                "Cprime_DISCLIMA_to_BULLYING"
            ]
        ),

        abs(
            point[
                "c_total"
            ]
            -
            previous[
                "Ctotal_DISCLIMA_to_BULLYING"
            ]
        ),

        abs(
            point[
                "indirect"
            ]
            -
            previous[
                "IndirectProduct_AxB"
            ]
        ),
    ]


    max_point_difference = max(
        point_differences
    )


    if max_point_difference > 1e-8:

        raise RuntimeError(

            f"{country}: point estimator does not "
            f"reproduce Phase 4F. "
            f"Max difference={max_point_difference}"
        )


    # --------------------------------------------------------
    # Deterministic system-specific seed
    # --------------------------------------------------------

    country_seed = (

        BASE_SEED

        +

        zlib.crc32(
            country.encode(
                "utf-8"
            )
        )
    ) % (
        2 ** 32
    )


    rng = np.random.default_rng(
        country_seed
    )


    bootstrap_a = np.empty(
        N_BOOTSTRAPS
    )


    bootstrap_b = np.empty(
        N_BOOTSTRAPS
    )


    bootstrap_cprime = np.empty(
        N_BOOTSTRAPS
    )


    bootstrap_ctotal = np.empty(
        N_BOOTSTRAPS
    )


    bootstrap_indirect = np.empty(
        N_BOOTSTRAPS
    )


    bootstrap_decomp_error = np.empty(
        N_BOOTSTRAPS
    )


    # ========================================================
    # School-cluster bootstrap
    # ========================================================

    for bootstrap_index in range(
        N_BOOTSTRAPS
    ):

        # ----------------------------------------------------
        # Sample G schools with replacement.
        #
        # Instead of physically duplicating students, convert
        # the number of times each school was sampled into a
        # multiplicity applied to W_FSTUWT.
        #
        # This is algebraically equivalent and much faster.
        # ----------------------------------------------------

        sampled_school_indices = rng.integers(

            low=0,

            high=G,

            size=G,
        )


        school_multiplicity = np.bincount(

            sampled_school_indices,

            minlength=G,
        )


        bootstrap_weights = (

            base_weights

            *

            school_multiplicity[
                school_codes
            ]
        )


        estimates = compute_path_estimates(

            raw=raw,

            bootstrap_weights=bootstrap_weights,
        )


        bootstrap_a[
            bootstrap_index
        ] = estimates[
            "a"
        ]


        bootstrap_b[
            bootstrap_index
        ] = estimates[
            "b"
        ]


        bootstrap_cprime[
            bootstrap_index
        ] = estimates[
            "c_prime"
        ]


        bootstrap_ctotal[
            bootstrap_index
        ] = estimates[
            "c_total"
        ]


        bootstrap_indirect[
            bootstrap_index
        ] = estimates[
            "indirect"
        ]


        bootstrap_decomp_error[
            bootstrap_index
        ] = estimates[
            "decomposition_error"
        ]


    # --------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------

    np.savez_compressed(

        checkpoint_path,

        country=
            country,

        N=
            N,

        N_schools=
            G,

        point_a=
            point[
                "a"
            ],

        point_b=
            point[
                "b"
            ],

        point_cprime=
            point[
                "c_prime"
            ],

        point_ctotal=
            point[
                "c_total"
            ],

        point_indirect=
            point[
                "indirect"
            ],

        max_point_difference=
            max_point_difference,

        bootstrap_a=
            bootstrap_a,

        bootstrap_b=
            bootstrap_b,

        bootstrap_cprime=
            bootstrap_cprime,

        bootstrap_ctotal=
            bootstrap_ctotal,

        bootstrap_indirect=
            bootstrap_indirect,

        bootstrap_decomp_error=
            bootstrap_decomp_error,
    )


    print(

        f"[{system_index:02d}/74] "
        f"{country}"
        f" | N={N}"
        f" | schools={G}"
        f" | indirect={point['indirect']:.5f}"
        f" | boot SE={np.std(bootstrap_indirect, ddof=1):.5f}"
        f" | CI=["
        f"{np.quantile(bootstrap_indirect, 0.025):.5f}, "
        f"{np.quantile(bootstrap_indirect, 0.975):.5f}]"
    )


    del (
        df,
        numeric,
        raw,
        base_weights,
        schools,
        bootstrap_a,
        bootstrap_b,
        bootstrap_cprime,
        bootstrap_ctotal,
        bootstrap_indirect,
        bootstrap_decomp_error,
    )


    gc.collect()


# ============================================================
# 10. Reassemble checkpoints
# ============================================================

system_rows = []

bootstrap_rows = []


for country in SYSTEMS:

    checkpoint_path = (

        CHECKPOINT_DIR

        /

        f"{country}.npz"
    )


    result = np.load(
        checkpoint_path,
        allow_pickle=True,
    )


    indirect_draws = result[
        "bootstrap_indirect"
    ]


    a_draws = result[
        "bootstrap_a"
    ]


    b_draws = result[
        "bootstrap_b"
    ]


    cprime_draws = result[
        "bootstrap_cprime"
    ]


    ctotal_draws = result[
        "bootstrap_ctotal"
    ]


    decomposition_draws = result[
        "bootstrap_decomp_error"
    ]


    point_indirect = float(
        result[
            "point_indirect"
        ]
    )


    boot_se = float(
        np.std(
            indirect_draws,
            ddof=1
        )
    )


    ci_low = float(
        np.quantile(
            indirect_draws,
            0.025
        )
    )


    ci_high = float(
        np.quantile(
            indirect_draws,
            0.975
        )
    )


    prop_negative = float(
        np.mean(
            indirect_draws
            <
            0
        )
    )


    system_rows.append(
        {

            "Country":
                country,

            "N":
                int(
                    result[
                        "N"
                    ]
                ),

            "N_Schools":
                int(
                    result[
                        "N_schools"
                    ]
                ),

            "A":
                float(
                    result[
                        "point_a"
                    ]
                ),

            "B":
                float(
                    result[
                        "point_b"
                    ]
                ),

            "Cprime":
                float(
                    result[
                        "point_cprime"
                    ]
                ),

            "Ctotal":
                float(
                    result[
                        "point_ctotal"
                    ]
                ),

            "Indirect":
                point_indirect,

            "BootstrapSE":
                boot_se,

            "BootstrapVariance":
                boot_se
                **
                2,

            "PercentileCI95_Low":
                ci_low,

            "PercentileCI95_High":
                ci_high,

            "BootstrapPropNegative":
                prop_negative,

            "CIEntirelyBelowZero":
                (
                    ci_high
                    <
                    0
                ),

            "MaxAbsBootstrapDecompositionError":
                float(
                    np.max(
                        np.abs(
                            decomposition_draws
                        )
                    )
                ),

            "PointReproductionMaxDifference":
                float(
                    result[
                        "max_point_difference"
                    ]
                ),
        }
    )


    for bootstrap_index in range(
        N_BOOTSTRAPS
    ):

        bootstrap_rows.append(
            {

                "Country":
                    country,

                "Bootstrap":
                    (
                        bootstrap_index
                        +
                        1
                    ),

                "A":
                    a_draws[
                        bootstrap_index
                    ],

                "B":
                    b_draws[
                        bootstrap_index
                    ],

                "Cprime":
                    cprime_draws[
                        bootstrap_index
                    ],

                "Ctotal":
                    ctotal_draws[
                        bootstrap_index
                    ],

                "Indirect":
                    indirect_draws[
                        bootstrap_index
                    ],
            }
        )


system_results = pd.DataFrame(
    system_rows
)


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


assert len(
    system_results
) == 74


assert len(
    bootstrap_results
) == (
    74
    *
    N_BOOTSTRAPS
)


# ============================================================
# 11. Save system and bootstrap outputs
# ============================================================

system_results.to_csv(

    OUT_DIR
    / "mediation_cluster_bootstrap_system_results.csv",

    index=False
)


bootstrap_results.to_csv(

    OUT_DIR
    / "mediation_cluster_bootstrap_draws.csv",

    index=False
)


# ============================================================
# 12. Bootstrap QA
# ============================================================

if (

    ~np.isfinite(
        system_results[
            "BootstrapSE"
        ]
    )
).any():

    raise RuntimeError(
        "Non-finite bootstrap SE found."
    )


if (

    system_results[
        "BootstrapSE"
    ]

    <=
    0

).any():

    raise RuntimeError(
        "Non-positive bootstrap SE found."
    )


max_reproduction_error = float(

    system_results[
        "PointReproductionMaxDifference"
    ]
    .max()
)


max_decomposition_error = float(

    system_results[
        "MaxAbsBootstrapDecompositionError"
    ]
    .max()
)


# ============================================================
# 13. RANDOM-EFFECTS META-ANALYSIS OF INDIRECT ASSOCIATION
# ============================================================

y = (

    system_results[
        "Indirect"
    ]
    .to_numpy(
        dtype=float
    )
)


variance = (

    system_results[
        "BootstrapVariance"
    ]
    .to_numpy(
        dtype=float
    )
)


X = np.ones(
    (
        len(
            system_results
        ),
        1
    )
)


meta_fit = fit_reml(

    y=y,

    variance=variance,

    X=X,
)


kh = modified_knapp_hartung(
    meta_fit
)


pooled_indirect = float(
    meta_fit[
        "beta"
    ][
        0
    ]
)


pooled_se = float(

    np.sqrt(
        kh[
            "covariance"
        ][
            0,
            0
        ]
    )
)


df_meta = int(
    kh[
        "df"
    ]
)


critical = float(

    student_t.ppf(
        0.975,
        df_meta
    )
)


ci_low = (

    pooled_indirect

    -

    critical

    *

    pooled_se
)


ci_high = (

    pooled_indirect

    +

    critical

    *

    pooled_se
)


t_value = (

    pooled_indirect

    /

    pooled_se
)


p_value = float(

    2

    *

    student_t.sf(
        abs(
            t_value
        ),
        df_meta
    )
)


tau2 = float(
    meta_fit[
        "tau2"
    ]
)


tau = float(
    np.sqrt(
        tau2
    )
)


# ============================================================
# 14. Cochran Q / I²
# ============================================================

fixed_weights = (
    1.0
    /
    variance
)


fixed_mean = float(

    np.sum(
        fixed_weights
        *
        y
    )

    /

    np.sum(
        fixed_weights
    )
)


Q = float(

    np.sum(

        fixed_weights

        *

        (
            y
            -
            fixed_mean
        )
        **
        2
    )
)


Q_df = (
    len(
        y
    )
    -
    1
)


Q_p = float(
    chi2.sf(
        Q,
        Q_df
    )
)


I2 = (

    max(
        0.0,

        (
            Q
            -
            Q_df
        )

        /

        Q
    )

    *

    100

    if Q
    >
    0

    else 0.0
)


# ============================================================
# 15. Random-effects prediction interval
# ============================================================

prediction_df = max(
    len(
        y
    )
    -
    2,
    1
)


prediction_critical = float(

    student_t.ppf(
        0.975,
        prediction_df
    )
)


prediction_se = float(

    np.sqrt(

        tau2

        +

        pooled_se
        **
        2
    )
)


prediction_low = (

    pooled_indirect

    -

    prediction_critical

    *

    prediction_se
)


prediction_high = (

    pooled_indirect

    +

    prediction_critical

    *

    prediction_se
)


# ============================================================
# 16. Meta-analysis result table
# ============================================================

meta_summary = pd.DataFrame(
    [
        {

            "Pathway":
                "DISCLIMA -> BELONG -> BEINGBULLIED",

            "N_Systems":
                len(
                    system_results
                ),

            "PooledIndirect":
                pooled_indirect,

            "ModifiedKH_SE":
                pooled_se,

            "T":
                t_value,

            "DF":
                df_meta,

            "P":
                p_value,

            "CI95_Low":
                ci_low,

            "CI95_High":
                ci_high,

            "Tau2":
                tau2,

            "Tau":
                tau,

            "Q":
                Q,

            "Q_df":
                Q_df,

            "Q_p":
                Q_p,

            "I2_Pct":
                I2,

            "PredictionInterval_Low":
                prediction_low,

            "PredictionInterval_High":
                prediction_high,

            "KH_RawScale":
                kh[
                    "KH_raw_scale"
                ],

            "KH_Scale":
                kh[
                    "KH_scale"
                ],

            "N_PointIndirectNegative":
                int(
                    (
                        system_results[
                            "Indirect"
                        ]
                        <
                        0
                    )
                    .sum()
                ),

            "N_BootstrapCIEntirelyBelowZero":
                int(
                    system_results[
                        "CIEntirelyBelowZero"
                    ]
                    .sum()
                ),
        }
    ]
)


meta_summary.to_csv(

    OUT_DIR
    / "mediation_indirect_random_effects_meta_analysis.csv",

    index=False
)


# ============================================================
# 17. Descriptive system summary
# ============================================================

system_summary = pd.DataFrame(
    {

        "Metric": [

            "Systems",

            "Point indirect < 0",

            "Bootstrap CI entirely < 0",

            "Median indirect",

            "Mean indirect",

            "Minimum indirect",

            "Maximum indirect",

            "Median bootstrap SE",

            "Minimum bootstrap negative proportion",

            "Median bootstrap negative proportion",

            "Maximum point reproduction error",

            "Maximum bootstrap decomposition error",
        ],

        "Value": [

            len(
                system_results
            ),

            (
                system_results[
                    "Indirect"
                ]
                <
                0
            )
            .sum(),

            system_results[
                "CIEntirelyBelowZero"
            ]
            .sum(),

            system_results[
                "Indirect"
            ]
            .median(),

            system_results[
                "Indirect"
            ]
            .mean(),

            system_results[
                "Indirect"
            ]
            .min(),

            system_results[
                "Indirect"
            ]
            .max(),

            system_results[
                "BootstrapSE"
            ]
            .median(),

            system_results[
                "BootstrapPropNegative"
            ]
            .min(),

            system_results[
                "BootstrapPropNegative"
            ]
            .median(),

            max_reproduction_error,

            max_decomposition_error,
        ],
    }
)


system_summary.to_csv(

    OUT_DIR
    / "mediation_cluster_bootstrap_summary.csv",

    index=False
)


# ============================================================
# 18. Systems with weakest bootstrap evidence
# ============================================================

weakest_systems = (

    system_results
    .sort_values(

        [

            "BootstrapPropNegative",

            "PercentileCI95_High",

        ],

        ascending=[
            True,
            False,
        ]
    )
    [
        [
            "Country",

            "N",

            "N_Schools",

            "Indirect",

            "BootstrapSE",

            "PercentileCI95_Low",

            "PercentileCI95_High",

            "BootstrapPropNegative",

            "CIEntirelyBelowZero",
        ]
    ]
    .head(
        15
    )
)


# ============================================================
# 19. Print QA
# ============================================================

print("\n")
print("=" * 125)
print("SCHOOL-CLUSTER BOOTSTRAP QA")
print("=" * 125)


print(
    "Systems:",
    len(
        system_results
    )
)


print(
    "Bootstrap draws:",
    len(
        bootstrap_results
    )
)


print(
    "Expected bootstrap draws:",
    74
    *
    N_BOOTSTRAPS
)


print(
    "Maximum Phase-4F point reproduction error:",
    max_reproduction_error
)


print(
    "Maximum bootstrap decomposition error:",
    max_decomposition_error
)


print(
    "Non-finite bootstrap SEs:",
    int(
        (
            ~np.isfinite(
                system_results[
                    "BootstrapSE"
                ]
            )
        )
        .sum()
    )
)


# ============================================================
# 20. System-level result summary
# ============================================================

print("\n")
print("=" * 125)
print("SYSTEM INDIRECT-ASSOCIATION SUMMARY")
print("=" * 125)


print(
    system_summary
    .round(
        7
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Random-effects meta-analysis
# ============================================================

print("\n")
print("=" * 125)
print("RANDOM-EFFECTS META-ANALYSIS OF INDIRECT ASSOCIATION")
print("=" * 125)


print(
    meta_summary
    .round(
        7
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 22. Weakest system bootstrap support
# ============================================================

print("\n")
print("=" * 125)
print("SYSTEMS WITH WEAKEST BOOTSTRAP SUPPORT")
print("=" * 125)


print(
    weakest_systems
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 23. Final status
# ============================================================

print("\n")
print("=" * 125)
print("PHASE 4G1 INDIRECT-ASSOCIATION ANALYSIS COMPLETE")
print("=" * 125)


print(
    "\nInterpretation remains explicitly noncausal:"
)


print(
    "This estimates a cross-sectional indirect association "
    "consistent with the pathway "
    "DISCLIMA -> BELONG -> BEINGBULLIED."
)


print(
    "\nIt does NOT establish temporal or causal mediation."
)

Systems: 74
Bootstrap replicates per system: 1000
[01/74] ALB | N=5814 | schools=326 | indirect=-0.03535 | boot SE=0.00382 | CI=[-0.04308, -0.02821]
[02/74] ARE | N=16597 | schools=750 | indirect=-0.06650 | boot SE=0.00419 | CI=[-0.07487, -0.05840]
[03/74] ARG | N=7420 | schools=445 | indirect=-0.04054 | boot SE=0.00616 | CI=[-0.05300, -0.02913]
[04/74] AUS | N=10439 | schools=716 | indirect=-0.05325 | boot SE=0.00510 | CI=[-0.06233, -0.04268]
[05/74] AUT | N=5153 | schools=275 | indirect=-0.03411 | boot SE=0.00454 | CI=[-0.04340, -0.02579]
[06/74] BEL | N=7047 | schools=269 | indirect=-0.02184 | boot SE=0.00407 | CI=[-0.03020, -0.01425]
[07/74] BGR | N=3517 | schools=196 | indirect=-0.04320 | boot SE=0.00576 | CI=[-0.05438, -0.03212]
[08/74] BIH | N=5167 | schools=212 | indirect=-0.02339 | boot SE=0.00453 | CI=[-0.03267, -0.01495]
[09/74] BLR | N=5250 | schools=233 | indirect=-0.03835 | boot SE=0.00331 | CI=[-0.04501, -0.03201]
[10/74] BRA | N=6169 | schools=559 | indirect=-0.03382 | 